<a href="https://colab.research.google.com/github/AlejoPatigno/MIL/blob/main/MIL_classificacion_segmentacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teoria

Vamos a tener las siguientes estructuras para el espacio de instancias, el espacio de las bolsas y el espacio de las etiquetas respectivamente.

$\mathbb{X} \subseteq \mathbb{R}^d$

$B_i = \{\mathbf{x}_{ij}\}_{j=1}^{m_i} \subset \mathbb{X}$

$y_i \in \{0,1,...,n\}$

Donde d es la dimension de cada una de las instancias, $m_i$ es el tamaño de la bolsa o, de forma más coloquial, la cantidad de instancias que hay en la bolsa i, y n lo tomarémos como la cantidad de etiquetas que puede tener cada bolsa. Luego, vamos a definir el dataset de la siguiente forma:

$\mathcal{D} = \{(B_i, y_i)\}_{i=1}^N$

donde N es la cantidad de bolsas dentro del dataset. Luego se definen las siguientes funciones

$f_\theta : \mathbb{X} → f_θ(\mathbb{X})$ where $f_θ(x_{ij}) = h_{ij}$

$g: \{\{h_{ij}\}_{j=1}^{m_i}\} _{i=1}^{N}→ g(\{\{h_{ij}\}_{j=1}^{m_i}\} _{i=1}^{N})$ where $ g(\{h_{ij}\}_{j=1}^{m_i}) = \mathbf{z}_i$

$\sigma: Z \rightarrow Y$ where  $\sigma(\mathbf{z}_i) = \hat{y}_i$

Definimos como Z el conjunto de todos los $\mathbf{z}_i$ y $Y\subseteq \{0,1,...,n\}$ al conjunto de todos los $\hat{y}_i$ o etiquetas predichas.


## mi-Net

En mi-Net $f_θ(\mathbb{X}) = \bar{Y} \subseteq \{0,1,...,n\}$, $f_θ$ es un red densa y $g$ en este caso se puede tomar con varios funciones de agregación, sin embargo, nos centraremos en 2.



## MI-Net

En MI-Net $f_θ(\mathbb{X}) \subseteq \mathbb{R}^{d'}$, $f_θ$ es un red densa y $g(\{\{h_{ij}\}_{j=1}^{m_i}\} _{i=1}^{N}) \subseteq \mathbb{R}^{d''} $ y así se hace una clasificación de la bolsa pasandolo por la función $\sigma$.

# Clasificación



In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random

# --- 1) Dataset MIL sobre MNIST ---
class MNISTBags(Dataset):
    def __init__(self, train=True, bag_size=10, num_bags=10000):
        self.ds = datasets.MNIST(root="./data", train=train, download=True,
                                 transform=transforms.ToTensor())
        self.bag_size, self.num_bags = bag_size, num_bags
        # Índices por dígito para muestrear rápido
        self.by_digit = {d: [i for i, (_, y) in enumerate(self.ds) if y == d] for d in range(10)}

    def __len__(self): return self.num_bags

    def __getitem__(self, _):
        # etiqueta de bolsa: con prob 0.5 generamos bolsa positiva (con al menos un '9')
        make_pos = random.random() < 0.5
        idxs = []
        if make_pos:
            idxs.append(random.choice(self.by_digit[9]))  # Garantiza 1 '9'
            while len(idxs) < self.bag_size:
                d = random.randint(0, 8)                   # no-9
                idxs.append(random.choice(self.by_digit[d]))
            y = 1
        else:
            while len(idxs) < self.bag_size:
                d = random.randint(0, 8)                   # solo no-9
                idxs.append(random.choice(self.by_digit[d]))
            y = 0
        random.shuffle(idxs)
        xs = torch.stack([self.ds[i][0] for i in idxs], dim=0)  # [n,1,28,28]
        return xs, torch.tensor([y], dtype=torch.float32)

# --- 2) Codificador por instancia (CNN) ---
class InstanceEncoder(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Linear(32 * 5 * 5, d)

    def forward(self, x):  # x: [n,1,28,28]
        z = self.conv(x)                       # [n,32,5,5]
        z = z.view(z.size(0), -1)              # [n,800]
        u = F.relu(self.fc(z))                 # [n,d]
        return u

import numpy as np

def confusion_from_binary(y_true, y_pred):
    """
    y_true, y_pred: arrays 0/1 de la misma forma.
    Devuelve TP, FP, FN, TN como escalares.
    """
    try:
        y_true = y_true.cpu().numpy()
        y_pred = y_pred.cpu().numpy()
    except:
      y_true = y_true.astype(np.int32)
      y_pred = y_pred.astype(np.int32)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    return tp, fp, fn, tn

def iou_score(tp, fp, fn, eps=1e-7):
    """
    IoU (Jaccard) para la clase positiva.
    IoU = TP / (TP + FP + FN)
    """
    return tp / (tp + fp + fn + eps)

def dice_score(tp, fp, fn, eps=1e-7):
    """
    Coeficiente de Dice = 2*TP / (2*TP + FP + FN)
    """
    return 2.0 * tp / (2.0 * tp + fp + fn + eps)

def precision_score(tp, fp, fn, tn, eps=1e-7):
    """
    Precision = TP / (TP + FP)
    """
    return tp / (tp + fp + eps)

def recall_score(tp, fp, fn, tn, eps=1e-7):
    """
    Recall / Sensibilidad = TP / (TP + FN)
    """
    return tp / (tp + fn + eps)

def specificity_score(tp, fp, fn, tn, eps=1e-7):
    """
    Especificidad = TN / (TN + FP)
    """
    return tn / (tn + fp + eps)

def balanced_accuracy(tp, fp, fn, tn, eps=1e-7):
    """
    Balanced Accuracy = (Recall + Specificity) / 2
    """
    rec = recall_score(tp, fp, fn, tn, eps)
    spec = specificity_score(tp, fp, fn, tn, eps)
    return 0.5 * (rec + spec)




In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random

# --- 1) Dataset MIL sobre MNIST ---
class MNISTBags(Dataset):
    def __init__(self, train=True, bag_size=10, num_bags=10000):
        self.ds = datasets.MNIST(root="./data", train=train, download=True,
                                 transform=transforms.ToTensor())
        self.bag_size, self.num_bags = bag_size, num_bags
        # Índices por dígito para muestrear rápido
        self.by_digit = {d: [i for i, (_, y) in enumerate(self.ds) if y == d] for d in range(10)}

    def __len__(self):
        return self.num_bags

    def __getitem__(self, _):
        # etiqueta de bolsa: con prob 0.5 generamos bolsa positiva (con al menos un '9')
        make_pos = random.random() < 0.5
        idxs = []
        if make_pos:
            idxs.append(random.choice(self.by_digit[9]))  # Garantiza 1 '9'
            while len(idxs) < self.bag_size:
                d = random.randint(0, 8)                   # no-9
                idxs.append(random.choice(self.by_digit[d]))
            y = 1
        else:
            while len(idxs) < self.bag_size:
                d = random.randint(0, 8)                   # solo no-9
                idxs.append(random.choice(self.by_digit[d]))
            y = 0
        random.shuffle(idxs)
        xs = torch.stack([self.ds[i][0] for i in idxs], dim=0)  # [n,1,28,28]
        return xs, torch.tensor([y], dtype=torch.float32)

# --- 2) Codificador por instancia (CNN) ---
class InstanceEncoder(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc = nn.Linear(32 * 5 * 5, d)

    def forward(self, x):  # x: [n,1,28,28]
        z = self.conv(x)                       # [n,32,5,5]
        z = z.view(z.size(0), -1)              # [n,800]
        u = F.relu(self.fc(z))                 # [n,d]
        return u

# --- 3) Agregador con atención + cabeza de bolsa ---
class AttentionMIL(nn.Module):
    def __init__(self, d=128):
        super().__init__()
        self.enc = InstanceEncoder(d)
        self.attn_V = nn.Linear(d, d)
        self.attn_w = nn.Linear(d, 1, bias=False)
        self.cls = nn.Linear(d, 1)

    def forward(self, bag):  # bag: [n,1,28,28]
        U = self.enc(bag)                         # [n,d]
        A = torch.tanh(self.attn_V(U))           # [n,d]
        alpha = torch.softmax(self.attn_w(A).squeeze(-1), dim=0)  # [n]
        phi = (alpha.unsqueeze(0) @ U).squeeze(0)                  # [d]

        # Logit de la bolsa
        logit_bag = self.cls(phi).view(1)        # [1]

        # Logits por instancia (misma cabeza lineal aplicada a cada embedding)
        inst_logits = self.cls(U).squeeze(-1)    # [n]

        # Devolvemos también los logits por instancia para la pérdida Top-k
        return logit_bag, alpha, inst_logits

# --- 4) Entrenamiento con pérdida de bolsa + Top-k BCE ---
def train_epoch(model, train_loader, test_loader, opt,
                device="cpu", top_k_ratio=0.3, lambda_topk=1.0):

    bce = nn.BCEWithLogitsLoss()

    # --- Fase de entrenamiento ---
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    # correct = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)    # x: [1, n, 1,28,28], y: [1,1]
        bag = x[0]                           # [n,1,28,28]

        logit_bag, alpha, inst_logits = model(bag)

        # Pérdida a nivel de bolsa
        loss_bag = bce(logit_bag.view_as(y), y)

        # Pérdida Top-k a nivel de instancia
        # k proporcional al tamaño de la bolsa
        n_inst = inst_logits.size(0)
        k = max(1, int(top_k_ratio * n_inst))
        k = min(k, n_inst)

        # Seleccionamos las k instancias con mayor peso de atención
        _, idx = torch.topk(alpha, k=k)           # índices [k]
        topk_logits = inst_logits[idx]            # [k]

        # Etiqueta de la bolsa replicada k veces
        y_scalar = y.float().mean()               # escalar 0. o 1.
        y_topk = y_scalar.expand_as(topk_logits)  # [k]

        loss_topk = bce(topk_logits, y_topk)

        # Combinación de pérdidas
        loss = loss_bag + lambda_topk * loss_topk

        opt.zero_grad()
        loss.backward()
        opt.step()

        loss_sum += loss.item()
        total += 1

        # Métrica de bolsa
        pred = (torch.sigmoid(logit_bag) >= 0.5).float()
        correct += (pred.view_as(y) == y).sum().item()

    train_acc = correct / total
    train_loss = loss_sum / total

    # --- Fase de evaluación ---
    model.eval()
    total_test, correct_test, loss_sum_test = 0, 0, 0.0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            bag = x[0]

            logit_bag, alpha, inst_logits = model(bag)

            # Mantenemos el mismo criterio de pérdida en validación
            loss_bag = bce(logit_bag.view_as(y), y)

            n_inst = inst_logits.size(0)
            k = max(1, int(top_k_ratio * n_inst))
            k = min(k, n_inst)

            _, idx = torch.topk(alpha, k=k)
            topk_logits = inst_logits[idx]

            y_scalar = y.float().mean()
            y_topk = y_scalar.expand_as(topk_logits)

            loss_topk = bce(topk_logits, y_topk)
            loss = loss_bag + lambda_topk * loss_topk

            loss_sum_test += loss.item()
            total_test += 1

            pred = (torch.sigmoid(logit_bag) >= 0.5).float()
            correct_test += (pred.view_as(y) == y).sum().item()

    test_acc = correct_test / total_test
    test_loss = loss_sum_test / total_test

    return train_acc, train_loss, test_acc, test_loss

# Uso:
train_loader = DataLoader(MNISTBags(train=True, bag_size=10, num_bags=4000),
                          batch_size=1, shuffle=True)
test_loader  = DataLoader(MNISTBags(train=False, bag_size=10, num_bags=1000),
                          batch_size=1)

device = "cpu"
model = AttentionMIL(d=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    acc, loss, acc_test, loss_test = train_epoch(
        model, train_loader, test_loader, opt,
        device=device,
        top_k_ratio=0.4,   # k ≈ 0.3 * bag_size
        lambda_topk=0.0    # peso del término Top-k
    )
    print(f"Epoch {epoch}: acc={acc:.3f}, loss={loss:.3f}, "
          f"acc_test={acc_test:.3f}, loss_test={loss_test:.3f}")


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.95MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 162kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.4MB/s]


Epoch 0: acc=0.735, loss=0.449, acc_test=0.925, loss_test=0.193
Epoch 1: acc=0.945, loss=0.157, acc_test=0.956, loss_test=0.127
Epoch 2: acc=0.966, loss=0.106, acc_test=0.966, loss_test=0.092
Epoch 3: acc=0.969, loss=0.094, acc_test=0.979, loss_test=0.085
Epoch 4: acc=0.972, loss=0.095, acc_test=0.970, loss_test=0.088
Epoch 5: acc=0.977, loss=0.070, acc_test=0.975, loss_test=0.068
Epoch 6: acc=0.976, loss=0.076, acc_test=0.968, loss_test=0.092
Epoch 7: acc=0.980, loss=0.068, acc_test=0.959, loss_test=0.115
Epoch 8: acc=0.982, loss=0.067, acc_test=0.963, loss_test=0.110
Epoch 9: acc=0.983, loss=0.060, acc_test=0.974, loss_test=0.068


In [ ]:
# Uso:
train_loader = DataLoader(MNISTBags(train=True, bag_size=100, num_bags=400),
                          batch_size=1, shuffle=True)
test_loader  = DataLoader(MNISTBags(train=False, bag_size=100, num_bags=100),
                          batch_size=1)

device = "cpu"
model = AttentionMIL(d=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    acc, loss, acc_test, loss_test = train_epoch(
        model, train_loader, test_loader, opt,
        device=device,
        top_k_ratio=0.4,   # k ≈ 0.3 * bag_size
        lambda_topk=0.0    # peso del término Top-k
    )
    print(f"Epoch {epoch}: acc={acc:.3f}, loss={loss:.3f}, "
          f"acc_test={acc_test:.3f}, loss_test={loss_test:.3f}")


Epoch 0: acc=0.465, loss=0.695, acc_test=0.500, loss_test=0.693
Epoch 1: acc=0.505, loss=0.694, acc_test=0.480, loss_test=0.693
Epoch 2: acc=0.500, loss=0.693, acc_test=0.450, loss_test=0.695
Epoch 3: acc=0.510, loss=0.693, acc_test=0.500, loss_test=0.693
Epoch 4: acc=0.482, loss=0.694, acc_test=0.560, loss_test=0.692
Epoch 5: acc=0.500, loss=0.693, acc_test=0.450, loss_test=0.694
Epoch 6: acc=0.502, loss=0.693, acc_test=0.540, loss_test=0.693
Epoch 7: acc=0.465, loss=0.693, acc_test=0.480, loss_test=0.693
Epoch 8: acc=0.510, loss=0.693, acc_test=0.470, loss_test=0.694
Epoch 9: acc=0.492, loss=0.693, acc_test=0.530, loss_test=0.693


In [ ]:
import numpy as np
import scipy.io as sio
from sklearn.model_selection import KFold
import tensorflow as tf
from keras.layers import Layer
from keras import backend as K
from keras import activations, initializers, regularizers

import numpy as np
from random import shuffle

# save batches, per batch contains instance features of a bag and bag label
def convertToBatch(bags):
    """Convert to batch format.
    Parameters
    -----------------
    bags : list
        A list contains instance features of bags and bag labels.
    Return
    -----------------
    data_set : list
        Convert dataset to batch format(instance features, bag label).
    """
    batch_num = len(bags)
    data_set = []
    for ibag, bag in enumerate(bags):
        # Flatten the image instances: from (N, 1, 28, 28) to (N, 784)
        flattened_instances = bag[0].view(bag[0].size(0), -1) # Convert PyTorch tensor to 2D
        batch_data = np.asarray(flattened_instances, dtype='float32') # Convert to numpy array
        batch_label = np.asarray(bag[1])
        data_set.append((batch_data, batch_label))
    return data_set

def bag_loss(y_true, y_pred):
    """Compute binary crossentropy loss of predicting bag loss.
    Parameters
    ---------------------
    y_true : Tensor (N x 1)
        GroundTruth of bag.
    y_pred : Tensor (1 X 1)
        Prediction score of bag.
    Return
    ---------------------
    acc : Tensor (1 x 1)
        Binary Crossentropy loss of predicting bag label.
    """
    # Use tf functions directly
    y_true_tf = tf.convert_to_tensor(y_true, dtype=tf.float32)
    y_pred_tf = tf.convert_to_tensor(y_pred, dtype=tf.float32)

    y_true_mean = tf.reduce_mean(y_true_tf, axis=0, keepdims=False)
    y_pred_mean = tf.reduce_mean(y_pred_tf, axis=0, keepdims=False)

    loss = tf.keras.losses.binary_crossentropy(y_true_mean, y_pred_mean)
    return loss

def max_pooling(x):
    """Max Pooling to obtain aggregation.
    Parameters
    ---------------------
    x : Tensor (N x d)
        Input data to do max-pooling,
        where N is number of instances in one bag,
        and d is dimension of instance feature
        (when d = 1, x means instance scores; when d > 1, x means instance representations).
    Return
    ---------------------
    output : Tensor (1 x d)
        Output of max-pooling,
        where d is dimension of instance feature
        (when d = 1, the output means bag score; when d > 1, the output means bag representation).
    """
    output = tf.reduce_max(x, axis=0, keepdims=True)
    return output

def mean_pooling(x):
    """Mean Pooling to obtain aggregation.
    Parameters
    ---------------------
    x : Tensor (N x d)
        Input data to do mean-pooling,
        where N is number of instances in one bag,
        and d is dimension of instance feature
        (when d = 1, x means instance scores; when d > 1, x means instance representations).
    Return
    ---------------------
    output : Tensor (1 x d)
        Output of mean-pooling,
        where d is dimension of instance feature
        (when d = 1, the output means bag score; when d > 1, the output means bag representation).
    """
    output = tf.reduce_mean(x, axis=0, keepdims=True)
    return output

def LSE_pooling(x):
    """LSE Pooling to obtain aggregation.
    Do LSE(log-sum-exp) pooling, like LSE(x1, x2, x3) = log(exp(x1)+exp(x2)+exp(x3)).
    Parameters
    ---------------------
    x : Tensor (N x d)
        Input data to do LSE-pooling,
        where N is number of instances in one bag,
        and d is dimension of instance feature
        (when d = 1, x means instance scores; when d > 1, x means instance representations).
    Return
    ---------------------
    output : Tensor (1 x d)
        Output of LSE-pooling,
        where d is dimension of instance feature
        (when d = 1, the output means bag score; when d > 1, the output means bag representation).
    """
    output = tf.math.log(tf.reduce_mean(tf.math.exp(x), axis=0, keepdims=True))
    return output

def choice_pooling(x, pooling_mode):
    """Choice the pooling mode
    Parameters
    -------------------
    x : Tensor (N x d)
        Input data to do MIL-pooling,
        where N is number of instances in one bag,
        and d is dimension of instance feature
        (when d = 1, x means instance scores; when d > 1, x means instance representations).
    pooling_mode : string
        Choice the pooling mode for MIL pooling.
    Return
    --------------------
    output : Tensor (1 x d)
            Output of MIL-pooling,
            where d is dimension of instance feature
            (when d = 1, the output means bag score; when d > 1, the output means bag representation).
    """
    if pooling_mode == 'max':
        return max_pooling(x)
    if pooling_mode == 'lse':
        return LSE_pooling(x)
    if pooling_mode == 'ave':
        return mean_pooling(x)

# Removed the redundant load_dataset function from this cell.

class Feature_pooling(Layer):
    """
    Feature pooling layer

    This layer contains a MIL pooling and a FC layer which only has one neural with
    sigmoid activation. The input of this layer is instance features. Via MIL pooling,
    we aggregate instance features to bag features. Finally, we obtain bag score by
    this FC layer with only one neural and sigmoid activation
    This layer is used in MI-Net and MI-Net with DS.

    # Arguments
        output_dim: Positive integer, dimensionality of the output space
        kernel_initializer: Initializer of the `kernel` weights matrix
        bias_initializer: Initializer of the `bias` weights
        kernel_regularizer: Regularizer function applied to the `kernel` weights matrix
        bias_regularizer: Regularizer function applied to the `bias` weights
        use_bias: Boolean, whether use bias or not
        pooling_mode: A string,
                      the mode of MIL pooling method, like 'max' (max pooling),
                      'ave' (average pooling), 'lse' (log-sum-exp pooling)

    # Input shape
        2D tensor with shape: (batch_size, input_dim)
    # Output shape
        2D tensor with shape: (1, units)
    """
    def __init__(self, output_dim, kernel_initializer='glorot_uniform', bias_initializer='zeros',
                    kernel_regularizer=None, bias_regularizer=None,
                    use_bias=True, pooling_mode='max', **kwargs):
        self.output_dim = output_dim
        self.pooling_mode = pooling_mode

        self.kernel_initializer = initializers.get(kernel_initializer)
        self.bias_initializer = initializers.get(bias_initializer)
        self.kernel_regularizer = regularizers.get(kernel_regularizer)
        self.bias_regularizer = regularizers.get(bias_regularizer)

        self.use_bias = use_bias
        super(Feature_pooling, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) == 2
        input_dim = input_shape[1]

        self.kernel = self.add_weight(shape=(input_dim, self.output_dim),
                                        initializer=self.kernel_initializer,
                                        name='kernel',
                                        regularizer=self.kernel_regularizer,
                                        trainable=True)

        if self.use_bias:
            self.bias = self.add_weight(shape=(self.output_dim,),
                                        initializer=self.bias_initializer,
                                        name='bias',
                                        regularizer=self.bias_regularizer,
                                        trainable=True)
        else:
            self.bias = None

        self.input_built = True

    def call(self, x, mask=None):
        n, d = x.shape

        # do-pooling operator
        x =choice_pooling(x, self.pooling_mode)

        # compute bag-level score
        output = tf.matmul(x, self.kernel)
        if self.use_bias:
            # Corrected: Use direct addition or tf.add instead of K.bias_add
            output = output + self.bias

        # sigmoid
        output = tf.keras.activations.sigmoid(output)

        return output

    def compute_output_shape(self, input_shape):
        shape = list(input_shape)
        assert len(shape) == 2
        shape[1] = self.output_dim
        return tuple(shape)

    def get_config(self):
        config = {
            'output_dim': self.output_dim,
            'kernel_initializer': initializers.serialize(self.kernel.initializer),
            'bias_initializer': initializers.serialize(self.bias_initializer),
            'kernel_regularizer': regularizers.serialize(self.kernel_regularizer),
            'bias_regularizer': regularizers.serialize(self.bias_regularizer),
            'use_bias': self.use_bias,
            'pooling_mode': self.pooling_mode
        }
        base_config = super(Feature_pooling, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

def bag_accuracy(y_true, y_pred):
    """Compute accuracy of one bag.
    Parameters
    ---------------------
    y_true : Tensor (N x 1)
        GroundTruth of bag.
    y_pred : Tensor (1 X 1)
        Prediction score of bag.
    Return
    ---------------------
    acc : Tensor (1 x 1)
        Accuracy of bag label prediction.
    """
    # Use tf functions directly
    y_true_tf = tf.convert_to_tensor(y_true, dtype=tf.float32)
    y_pred_tf = tf.convert_to_tensor(y_pred, dtype=tf.float32)

    y_true_mean = tf.reduce_mean(y_true_tf, axis=0, keepdims=False)
    y_pred_mean = tf.reduce_mean(y_pred_tf, axis=0, keepdims=False)

    rounded_pred = tf.round(y_pred_mean)
    acc = tf.cast(tf.equal(y_true_mean, rounded_pred), dtype=tf.float32)
    return acc

In [ ]:
import numpy as np
import sys
import time
import random
from random import shuffle
import argparse

from keras.models import Model
from keras.optimizers import SGD
from keras.regularizers import l2
from keras.layers import Input, Dense, Layer, Dropout

# Import KFold for cross-validation and DataLoader (if not already in global scope)
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader # Assuming DataLoader is needed for MNISTBags

def parse_args():
    """Parse input arguments.
    Parameters
    -------------------
    No parameters.
    Returns
    -------------------
    args: argparser.Namespace class object
        An argparse.Namespace class object contains experimental hyper-parameters.
    """
    parser = argparse.ArgumentParser(description='Train a MI-Net')
    parser.add_argument('--dataset', dest='dataset',
                        help='dataset to train on, like musk1 or fox',
                        default=None, type=str)
    parser.add_argument('--pooling', dest='pooling_mode',
                        help='mode of MIL pooling',
                        default='max', type=str)
    parser.add_argument('--lr', dest='init_lr',
                        help='initial learning rate',
                        default=5e-4, type=float)
    parser.add_argument('--decay', dest='weight_decay',
                        help='weight decay',
                        default=0.005, type=float)
    parser.add_argument('--momentum', dest='momentum',
                        help='momentum',
                        default=0.9, type=float)
    parser.add_argument('--epoch', dest='max_epoch',
                        help='number of epoch to train',
                        default=10, type=int)

    args = parser.parse_args([]) # Modified line
    return args

def test_eval(model, test_set):
    """Evaluate on testing set.
    Parameters
    -----------------
    model : keras.engine.training.Model object
        The training MI-Net model.
    test_set : list
        A list of testing set contains all training bags features and labels.
    Returns
    -----------------
    test_loss : float
        Mean loss of evaluating on testing set.
    test_acc : float
        Mean accuracy of evaluating on testing set.
    """
    num_test_batch = len(test_set)
    test_loss = np.zeros((num_test_batch, 1), dtype=float)
    test_acc = np.zeros((num_test_batch, 1), dtype=float)
    for ibatch, batch in enumerate(test_set):
        # Corrected: Pass batch[1] directly as target
        result = model.test_on_batch({'input':batch[0]}, batch[1])
        test_loss[ibatch] = result[0]
        test_acc[ibatch][0] = result[1]
    return np.mean(test_loss), np.mean(test_acc)

def train_eval(model, train_set):
    """Evaluate on training set.
    Parameters
    -----------------
    model : keras.engine.training.Model object
        The training MI-Net model.
    train_set : list
        A list of training set contains all training bags features and labels.
    Returns
    -----------------
    test_loss : float
        Mean loss of evaluating on traing set.
    test_acc : float
        Mean accuracy of evaluating on testing set.
    """
    num_train_batch = len(train_set)
    train_loss = np.zeros((num_train_batch, 1), dtype=float)
    train_acc = np.zeros((num_train_batch, 1), dtype=float)
    shuffle(train_set)
    for ibatch, batch in enumerate(train_set):
        # Corrected: Pass batch[1] directly as target
        result = model.train_on_batch({'input':batch[0]}, batch[1])
        train_loss[ibatch] = result[0]
        train_acc[ibatch][0] = result[1]
    return np.mean(train_loss), np.mean(train_acc)

def load_dataset(dataset_nm, n_folds):
    """Load data from file, do pre-processing, split it into train/test set.
    Parameters
    -----------------
    dataset_nm : string
        Name of dataset.
    n_folds : int
        Number of cross-validation folds.
    Returns
    -----------------
    datasets : list
        List contains split datasets for k-Fold cross-validation.
    """
    # Create an instance of MNISTBags and load all bags into a list
    mnist_dataset = MNISTBags(train=True, bag_size=100, num_bags=400)
    all_bags = [mnist_dataset[i] for i in range(len(mnist_dataset))]

    num_bag = len(all_bags) # This is the total number of bags

    # Correct KFold usage: n_splits instead of n_folds, and call .split() on indices
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=None)
    datasets = []
    for train_idx, test_idx in kf.split(range(num_bag)):
        dataset = {}
        # Use the generated indices to slice the `all_bags` list
        dataset['train'] = [all_bags[ibag] for ibag in train_idx]
        dataset['test'] = [all_bags[ibag] for ibag in test_idx]
        datasets.append(dataset)
    return datasets

def MI_Net(dataset):
    """Train and evaluate on MI-Net.
    Parameters
    -----------------
    dataset : dict
        A dictionary contains all dataset information. We split train/test by keys.
    Returns
    -----------------
    test_acc : float
        Testing accuracy of MI-Net.
    """
    # load data and convert type
    train_bags = dataset['train']
    test_bags = dataset['test']

    # convert bag to batch
    train_set = convertToBatch(train_bags)
    test_set = convertToBatch(test_bags)
    dimension = train_set[0][0].shape[1]

    # data: instance feature, n*d, n = number of training instance
    data_input = Input(shape=(dimension,), dtype='float32', name='input')

    # fully-connected
    fc1 = Dense(256, activation='relu', kernel_regularizer=l2(args.weight_decay))(data_input)
    fc2 = Dense(128, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc1)
    fc3 = Dense(64, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc2)

    # dropout
    dropout = Dropout(rate=0.5)(fc3)

    # features pooling
    fp = Feature_pooling(output_dim=1, kernel_regularizer=l2(args.weight_decay), pooling_mode=args.pooling_mode, name='fp')(dropout)

    model = Model(inputs=[data_input], outputs=[fp])
    sgd = SGD(learning_rate=args.init_lr, momentum=args.momentum, nesterov=True)
    # Fix: Directly pass loss and metrics for a single output model
    model.compile(loss=bag_loss, optimizer=sgd, metrics=[bag_accuracy])

    # train model
    t1 = time.time()
    num_batch = len(train_set)
    for epoch in range(args.max_epoch):
        train_loss, train_acc = train_eval(model, train_set)
        test_loss, test_acc = test_eval(model, test_set)
        print( 'epoch=', epoch, '  train_loss= {:.3f}'.format(train_loss), '  train_acc= {:.3f}'.format(train_acc), '  test_loss={:.3f}'.format(test_loss), '  test_acc= {:.3f}'.format(test_acc))
    t2 = time.time()
    print( 'run time:', (t2-t1) / 60, 'min')
    print( 'test_acc={:.3f}'.format(test_acc))

    return test_acc



args = parse_args()
print( 'Called with args:')
print( args)
# perform five times 10-fold cross-validation experiments
run = 1
n_folds = 3
acc = np.zeros((run, n_folds), dtype=float)
for irun in range(run):
    dataset = load_dataset(args.dataset, n_folds)
    for ifold in range(n_folds):
        print( 'run=', irun, '  fold=', ifold)
        acc[irun][ifold] = MI_Net(dataset[ifold])
print( 'MI-Net mean accuracy = ', np.mean(acc))
print( 'std = ', np.std(acc))


Called with args:
Namespace(dataset=None, pooling_mode='max', init_lr=0.0005, weight_decay=0.005, momentum=0.9, max_epoch=10)
run= 0   fold= 0
epoch= 0   train_loss= 3.967   train_acc= 0.530   test_loss=3.918   test_acc= 0.513
epoch= 1   train_loss= 3.878   train_acc= 0.513   test_loss=3.841   test_acc= 0.540
epoch= 2   train_loss= 3.817   train_acc= 0.548   test_loss=3.793   test_acc= 0.550
epoch= 3   train_loss= 3.766   train_acc= 0.557   test_loss=3.743   test_acc= 0.558
epoch= 4   train_loss= 3.719   train_acc= 0.560   test_loss=3.696   test_acc= 0.567
epoch= 5   train_loss= 3.674   train_acc= 0.573   test_loss=3.650   test_acc= 0.581
epoch= 6   train_loss= 3.628   train_acc= 0.588   test_loss=3.605   test_acc= 0.596
epoch= 7   train_loss= 3.583   train_acc= 0.602   test_loss=3.561   test_acc= 0.608
epoch= 8   train_loss= 3.539   train_acc= 0.614   test_loss=3.518   test_acc= 0.617
epoch= 9   train_loss= 3.497   train_acc= 0.621   test_loss=3.477   test_acc= 0.625
run time: 0.37037

In [ ]:
import sys
import time
from random import shuffle
import numpy as np
import argparse

from keras.models import Model
from keras.optimizers import SGD
from keras.regularizers import l2
from keras.layers import Input, Dense, Layer, Dropout
from keras.layers import Layer
from keras import backend as K
from keras import activations, initializers, regularizers
import tensorflow as tf # Import tensorflow for tf.matmul and tf.keras.activations.sigmoid

# Additions for MNISTBags and load_dataset dependencies
import torch
import torch.utils.data as data_utils
from torchvision import datasets, transforms
from sklearn.model_selection import KFold
import random # Imported explicitly for MNISTBags as it uses random.choice

# --- MNISTBags Class (copied from H-wMI_co_C6P) ---
class MNISTBags(data_utils.Dataset):
    def __init__(self, train=True, bag_size=10, num_bags=10000):
        self.ds = datasets.MNIST(root="./data", train=train, download=True,
                                 transform=transforms.ToTensor())
        self.bag_size, self.num_bags = bag_size, num_bags
        # Índices por dígito para muestrear rápido
        self.by_digit = {d: [i for i, (_, y) in enumerate(self.ds) if y == d] for d in range(10)}

    def __len__(self): return self.num_bags

    def __getitem__(self, _):
        # etiqueta de bolsa: con prob 0.5 generamos bolsa positiva (con al menos un '9')
        make_pos = random.random() < 0.5
        idxs = []
        if make_pos:
            idxs.append(random.choice(self.by_digit[9]))  # Garantiza 1 '9'
            while len(idxs) < self.bag_size:
                d = random.randint(0, 8)                   # no-9
                idxs.append(random.choice(self.by_digit[d]))
            y = 1
        else:
            while len(idxs) < self.bag_size:
                d = random.randint(0, 8)                   # solo no-9
                idxs.append(random.choice(self.by_digit[d]))
            y = 0
        random.shuffle(idxs)
        xs = torch.stack([self.ds[i][0] for i in idxs], dim=0)  # [n,1,28,28]
        return xs, torch.tensor([y], dtype=torch.float32)

# --- Pooling functions (copied from n8pSnjXOjur1) ---
def max_pooling(x):
    output = tf.reduce_max(x, axis=0, keepdims=True)
    return output

def mean_pooling(x):
    output = tf.reduce_mean(x, axis=0, keepdims=True)
    return output

def LSE_pooling(x):
    output = tf.math.log(tf.reduce_mean(tf.math.exp(x), axis=0, keepdims=True))
    return output

def choice_pooling(x, pooling_mode):
    if pooling_mode == 'max':
        return max_pooling(x)
    if pooling_mode == 'lse':
        return LSE_pooling(x)
    if pooling_mode == 'ave':
        return mean_pooling(x)

# --- bag_loss and bag_accuracy (copied from n8pSnjXOjur1) ---
def bag_loss(y_true, y_pred):
    y_true_tf = tf.convert_to_tensor(y_true, dtype=tf.float32)
    y_pred_tf = tf.convert_to_tensor(y_pred, dtype=tf.float32)
    y_true_mean = tf.reduce_mean(y_true_tf, axis=0, keepdims=False)
    y_pred_mean = tf.reduce_mean(y_pred_tf, axis=0, keepdims=False)
    loss = tf.keras.losses.binary_crossentropy(y_true_mean, y_pred_mean)
    return loss

def bag_accuracy(y_true, y_pred):
    y_true_tf = tf.convert_to_tensor(y_true, dtype=tf.float32)
    y_pred_tf = tf.convert_to_tensor(y_pred, dtype=tf.float32)
    y_true_mean = tf.reduce_mean(y_true_tf, axis=0, keepdims=False)
    y_pred_mean = tf.reduce_mean(y_pred_tf, axis=0, keepdims=False)
    rounded_pred = tf.round(y_pred_mean)
    acc = tf.cast(tf.equal(y_true_mean, rounded_pred), dtype=tf.float32)
    return acc

# --- convertToBatch (copied from n8pSnjXOjur1) ---
def convertToBatch(bags):
    batch_num = len(bags)
    data_set = []
    for ibag, bag in enumerate(bags):
        # Flatten the image instances: from (N, 1, 28, 28) to (N, 784)
        flattened_instances = bag[0].view(bag[0].size(0), -1) # Convert PyTorch tensor to 2D
        batch_data = np.asarray(flattened_instances, dtype='float32') # Convert to numpy array
        batch_label = np.asarray(bag[1])
        data_set.append((batch_data, batch_label))
    return data_set

# --- Score_pooling Class ---
class Score_pooling(Layer):
    def __init__(self, output_dim, kernel_initializer='glorot_uniform', bias_initializer='zeros',
                    kernel_regularizer=None, bias_regularizer=None,
                    use_bias=True, pooling_mode='max', **kwargs):
        self.output_dim = output_dim
        self.pooling_mode = pooling_mode

        self.kernel_initializer = initializers.get(kernel_initializer)
        self.bias_initializer = initializers.get(bias_initializer)
        self.kernel_regularizer = regularizers.get(kernel_regularizer)
        self.bias_regularizer = regularizers.get(bias_regularizer)

        self.use_bias = use_bias
        super(Score_pooling, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) == 2
        input_dim = input_shape[1]

        self.kernel = self.add_weight(shape=(input_dim, self.output_dim),
                                        initializer=self.kernel_initializer,
                                        name='kernel',
                                        regularizer=self.kernel_regularizer)

        if self.use_bias:
            self.bias = self.add_weight(shape=(self.output_dim,),
                                        initializer=self.bias_initializer,
                                        name='bias',
                                        regularizer=self.bias_regularizer)
        else:
            self.bias = None

        self.input_built = True

    def call(self, x, mask=None):
        n, d = x.shape

        # compute instance-level score
        x = tf.matmul(x, self.kernel)
        if self.use_bias:
            x = x + self.bias

        # sigmoid
        x = tf.keras.activations.sigmoid(x)

        # do-pooling operator
        output = choice_pooling(x, self.pooling_mode)

        return output

    def compute_output_shape(self, input_shape):
        shape = list(input_shape)
        assert len(shape) == 2
        shape[1] = self.output_dim
        return tuple(shape)

    def get_config(self):
        config = {
            'output_dim': self.output_dim,
            'kernel_initializer': initializers.serialize(self.kernel.initializer),
            'bias_initializer': initializers.serialize(self.bias_initializer),
            'kernel_regularizer': regularizers.serialize(self.kernel_regularizer),
            'bias_regularizer': regularizers.serialize(self.bias_regularizer),
            'use_bias': self.use_bias,
            'pooling_mode': self.pooling_mode
        }
        base_config = super(Score_pooling, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))

# --- parse_args function ---
def parse_args():
    parser = argparse.ArgumentParser(description='Train a mi-net')
    parser.add_argument('--dataset', dest='dataset',
                        help='dataset to train on, like musk1 or fox',
                        default=None, type=str)
    parser.add_argument('--pooling', dest='pooling_mode',
                        help='mode of MIL pooling',
                        default='max', type=str)
    parser.add_argument('--lr', dest='init_lr',
                        help='initial learning rate',
                        default=5e-4, type=float)
    parser.add_argument('--decay', dest='weight_decay',
                        help='weight decay',
                        default=0.005, type=float)
    parser.add_argument('--momentum', dest='momentum',
                        help='momentum',
                        default=0.9, type=float)
    parser.add_argument('--epoch', dest='max_epoch',
                        help='number of epoch to train',
                        default=10, type=int)

    # Modify this line to pass an empty list for Colab compatibility
    args = parser.parse_args([])
    return args

# --- test_eval function ---
def test_eval(model, test_set):
    num_test_batch = len(test_set)
    test_loss = np.zeros((num_test_batch, 1), dtype=float)
    test_acc = np.zeros((num_test_batch, 1), dtype=float)
    for ibatch, batch in enumerate(test_set):
        # Pass batch[1] directly as target, not as a dictionary with 'sp'
        result = model.test_on_batch({'input':batch[0]}, batch[1])
        test_loss[ibatch] = result[0]
        test_acc[ibatch] = result[1]
    return np.mean(test_loss), np.mean(test_acc)

# --- train_eval function ---
def train_eval(model, train_set):
    num_train_batch = len(train_set)
    train_loss = np.zeros((num_train_batch, 1), dtype=float)
    train_acc = np.zeros((num_train_batch, 1), dtype=float)
    shuffle(train_set)
    for ibatch, batch in enumerate(train_set):
        # Pass batch[1] directly as target, not as a dictionary with 'sp'
        result = model.train_on_batch({'input':batch[0]}, batch[1])
        train_loss[ibatch] = result[0]
        train_acc[ibatch] = result[1]
    return np.mean(train_loss), np.mean(train_acc)

# --- load_dataset function (copied from VhTtE6pCcLr7 and fixed) ---
def load_dataset(dataset_nm, n_folds):
    mnist_dataset = MNISTBags(train=True, bag_size=100, num_bags=400)
    all_bags = [mnist_dataset[i] for i in range(len(mnist_dataset))]

    num_bag = len(all_bags) # This is the total number of bags

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=None)
    kfold_data_splits = [] # Renamed 'datasets' to 'kfold_data_splits' to avoid shadowing
    for train_idx, test_idx in kf.split(range(num_bag)):
        dataset = {}
        # Use the generated indices to slice the `all_bags` list
        dataset['train'] = [all_bags[ibag] for ibag in train_idx]
        dataset['test'] = [all_bags[ibag] for ibag in test_idx]
        kfold_data_splits.append(dataset)
    return kfold_data_splits # Return the renamed variable


# --- mi_Net function ---
def mi_Net(dataset):
    """Train and evaluate on mi-Net.
    Parameters
    -----------------
    dataset : dict
        A dictionary contains all dataset information. We split train/test by keys.
    Returns
    -----------------
    test_acc : float
        Testing accuracy of mi-Net.
    """
    # load data and convert type
    train_bags = dataset['train']
    test_bags = dataset['test']

    # convert bag to batch
    train_set = convertToBatch(train_bags)
    test_set = convertToBatch(test_bags)
    dimension = train_set[0][0].shape[1]

    # data: instance feature, n*d, n = number of training instance
    data_input = Input(shape=(dimension,), dtype='float32', name='input')

    # fully-connected
    fc1 = Dense(256, activation='relu', kernel_regularizer=l2(args.weight_decay))(data_input)
    fc2 = Dense(128, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc1)
    fc3 = Dense(64, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc2)

    # dropout
    dropout = Dropout(rate=0.5)(fc3)

    # score pooling
    sp = Score_pooling(output_dim=1, kernel_regularizer=l2(args.weight_decay), pooling_mode=args.pooling_mode, name='sp')(dropout)

    model = Model(inputs=[data_input], outputs=[sp])
    sgd = SGD(learning_rate=args.init_lr, momentum=args.momentum, nesterov=True)
    # For a single output model, pass loss and metrics directly, not as a dictionary
    model.compile(loss=bag_loss, optimizer=sgd, metrics=[bag_accuracy])

    # train model
    t1 = time.time()
    num_batch = len(train_set)
    for epoch in range(args.max_epoch):
        train_loss, train_acc = train_eval(model, train_set)
        test_loss, test_acc = test_eval(model, test_set)
        print( 'epoch=', epoch, '  train_loss= {:.3f}'.format(train_loss), '  train_acc= {:.3f}'.format(train_acc), '  test_loss={:.3f}'.format(test_loss), '  test_acc= {:.3f}'.format(test_acc))
    t2 = time.time()
    print( 'run time:', (t2-t1) / 60.0, 'min')
    print( 'test_acc={:.3f}'.format(test_acc))

    return test_acc


args = parse_args()
print( 'Called with args:')
print( args)
# perform five times 10-fold cross-validation experiments
run = 1
n_folds = 3
acc = np.zeros((run, n_folds), dtype=float)
for irun in range(run):
    # This `dataset_list_of_dicts` variable is the list returned by `load_dataset`
    dataset_list_of_dicts = load_dataset(args.dataset, n_folds)
    for ifold in range(n_folds):
        print( 'run=', irun, '  fold=', ifold)
        acc[irun][ifold] = mi_Net(dataset_list_of_dicts[ifold])
print( 'mi-net mean accuracy = ', np.mean(acc))
print( 'std = ', np.std(acc))

Called with args:
Namespace(dataset=None, pooling_mode='max', init_lr=0.0005, weight_decay=0.005, momentum=0.9, max_epoch=10)
run= 0   fold= 0
epoch= 0   train_loss= 3.946   train_acc= 0.470   test_loss=3.876   test_acc= 0.500
epoch= 1   train_loss= 3.857   train_acc= 0.493   test_loss=3.835   test_acc= 0.491
epoch= 2   train_loss= 3.814   train_acc= 0.496   test_loss=3.793   test_acc= 0.500
epoch= 3   train_loss= 3.774   train_acc= 0.499   test_loss=3.753   test_acc= 0.494
epoch= 4   train_loss= 3.734   train_acc= 0.496   test_loss=3.713   test_acc= 0.496
epoch= 5   train_loss= 3.694   train_acc= 0.497   test_loss=3.674   test_acc= 0.495
epoch= 6   train_loss= 3.655   train_acc= 0.498   test_loss=3.635   test_acc= 0.500
epoch= 7   train_loss= 3.616   train_acc= 0.503   test_loss=3.597   test_acc= 0.503
epoch= 8   train_loss= 3.578   train_acc= 0.501   test_loss=3.560   test_acc= 0.500
epoch= 9   train_loss= 3.542   train_acc= 0.501   test_loss=3.523   test_acc= 0.503
run time: 0.36809

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.utils.data as data_utils
from torchvision import datasets, transforms


class MnistBags(data_utils.Dataset):
    def __init__(self, target_number=9, mean_bag_length=10, var_bag_length=1, num_bag=1000, seed=7, train=True):
        self.target_number = target_number
        self.mean_bag_length = mean_bag_length
        self.var_bag_length = var_bag_length
        self.num_bag = num_bag
        self.seed = seed
        self.train = train

        self.r = np.random.RandomState(seed)

        self.num_in_train = 60000
        self.num_in_test = 10000

        if self.train:
            self.train_bags_list, self.train_labels_list = self._form_bags()
        else:
            self.test_bags_list, self.test_labels_list = self._form_bags()

    def _form_bags(self):
        if self.train:
            train_loader = data_utils.DataLoader(datasets.MNIST('../datasets',
                                                                train=True,
                                                                download=True,
                                                                transform=transforms.Compose([
                                                                         transforms.ToTensor(),
                                                                         transforms.Normalize((0.1307,), (0.3081,))])),
                                                 batch_size=self.num_in_train,
                                                 shuffle=False)

            bags_list = []
            labels_list = []
            valid_bags_counter = 0
            label_of_last_bag = 0

            for batch_data in train_loader:
                numbers = batch_data[0]
                labels = batch_data[1]

            while valid_bags_counter < self.num_bag:
                bag_length = int(self.r.normal(self.mean_bag_length, self.var_bag_length, 1)[0]) # Fixed here
                if bag_length < 1:
                    bag_length = 1
                indices = torch.LongTensor(self.r.randint(0, self.num_in_train, bag_length))
                labels_in_bag = labels[indices]

                if (self.target_number in labels_in_bag) and (label_of_last_bag == 0):
                    labels_in_bag = labels_in_bag >= self.target_number
                    labels_list.append(labels_in_bag)
                    bags_list.append(numbers[indices])
                    label_of_last_bag = 1
                    valid_bags_counter += 1
                elif label_of_last_bag == 1:
                    index_list = []
                    bag_length_counter = 0
                    while bag_length_counter < bag_length:
                        index = torch.LongTensor(self.r.randint(0, self.num_in_train, 1))
                        label_temp = labels[index]
                        if label_temp.numpy()[0] != self.target_number:
                            index_list.append(index)
                            bag_length_counter += 1

                    index_list = np.array(index_list)
                    labels_in_bag = labels[index_list]
                    labels_in_bag = labels_in_bag >= self.target_number
                    labels_list.append(labels_in_bag)
                    bags_list.append(numbers[index_list])
                    label_of_last_bag = 0
                    valid_bags_counter += 1
                else:
                    pass

        else:
            test_loader = data_utils.DataLoader(datasets.MNIST('../datasets',
                                                               train=False,
                                                               download=True,
                                                               transform=transforms.Compose([
                                                                    transforms.ToTensor(),
                                                                    transforms.Normalize((0.1307,), (0.3081,))])),
                                                batch_size=self.num_in_test,
                                                shuffle=False)

            bags_list = []
            labels_list = []
            valid_bags_counter = 0
            label_of_last_bag = 0

            for batch_data in test_loader:
                numbers = batch_data[0]
                labels = batch_data[1]

            while valid_bags_counter < self.num_bag:
                bag_length = int(self.r.normal(self.mean_bag_length, self.var_bag_length, 1)[0]) # Fixed here
                if bag_length < 1:
                    bag_length = 1
                indices = torch.LongTensor(self.r.randint(0, self.num_in_test, bag_length))
                labels_in_bag = labels[indices]

                if (self.target_number in labels_in_bag) and (label_of_last_bag == 0):
                    labels_in_bag = labels_in_bag >= self.target_number
                    labels_list.append(labels_in_bag)
                    bags_list.append(numbers[indices])
                    label_of_last_bag = 1
                    valid_bags_counter += 1
                elif label_of_last_bag == 1:
                    index_list = []
                    bag_length_counter = 0
                    while bag_length_counter < bag_length:
                        index = torch.LongTensor(self.r.randint(0, self.num_in_test, 1))
                        label_temp = labels[index]
                        if label_temp.numpy()[0] != self.target_number:
                            index_list.append(index)
                            bag_length_counter += 1

                    index_list = np.array(index_list)
                    labels_in_bag = labels[index_list]
                    labels_in_bag = labels_in_bag >= self.target_number
                    labels_list.append(labels_in_bag)
                    bags_list.append(numbers[index_list])
                    label_of_last_bag = 0
                    valid_bags_counter += 1
                else:
                    pass

        return bags_list, labels_list

    def __len__(self):
        if self.train:
            return len(self.train_labels_list)
        else:
            return len(self.test_labels_list)

    def __getitem__(self, index):
        if self.train:
            bag = self.train_bags_list[index]
            label = [max(self.train_labels_list[index]), self.train_labels_list[index]]
        else:
            bag = self.test_bags_list[index]
            label = [max(self.test_labels_list[index]), self.test_labels_list[index]]

        return bag, label



to_pil = transforms.Compose([transforms.ToPILImage()])
kwargs = {}
batch_size = 1
train_loader = data_utils.DataLoader(MnistBags(target_number=9,
                                               mean_bag_length=10,
                                               var_bag_length=2,
                                               num_bag=100,
                                               seed=98,
                                               train=True),
                                     batch_size=batch_size,
                                     shuffle=False, **kwargs)
test_loader = data_utils.DataLoader(MnistBags(target_number=9,
                                              mean_bag_length=10,
                                              var_bag_length=2,
                                              num_bag=10,
                                              seed=98,
                                              train=False),
                                    batch_size=batch_size,
                                    shuffle=False, **kwargs)
len_bag_list = []
mnist_bags_train = 0
for batch_idx, data in enumerate(train_loader):
    plot_data = data[0].squeeze(0)
    len_bag_list.append(int(plot_data.size()[0]))
    # plot_data = data[0].squeeze(0)
    # num_instances = int(plot_data.size()[0])
    # print(data[1][0])
    # for i in range(num_instances):
    #     plt.subplot(num_instances, 1, i + 1)
    #     to_pil(plot_data[i, :, :, :]).show()
    # plt.show()
    if data[1][0][0] == 1:
        mnist_bags_train += 1
print('number of bags with 9(s): ', mnist_bags_train)
print('total number of bags', len(train_loader))
print(np.mean(len_bag_list), np.min(len_bag_list), np.max(len_bag_list))
len_bag_list = []
mnist_bags_test = 0
for batch_idx, data in enumerate(test_loader):
    plot_data = data[0].squeeze(0)
    len_bag_list.append(int(plot_data.size()[0]))
    if data[1][0][0] == 1:
        mnist_bags_test += 1
print('number of bags with 9(s): ', mnist_bags_test)
print('total number of bags', len(test_loader))
print(np.mean(len_bag_list), np.min(len_bag_list), np.max(len_bag_list))


100%|██████████| 9.91M/9.91M [00:01<00:00, 6.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 157kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.81MB/s]


number of bags with 9(s):  50
total number of bags 100
9.72 5 14
number of bags with 9(s):  5
total number of bags 10
10.6 9 13


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.feature_extractor_part1 = nn.Sequential(
            nn.Conv2d(1, 20, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(20, 50, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2)
        )

        self.feature_extractor_part2 = nn.Sequential(
            nn.Linear(50 * 4 * 4, self.M),
            nn.ReLU(),
        )

        self.attention = nn.Sequential(
            nn.Linear(self.M, self.L), # matrix V
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES) # matrix w (or vector w if self.ATTENTION_BRANCHES==1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.M*self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)
        # Add squeeze(2) to remove the unexpected singleton dimension if it exists
        if x.dim() == 5 and x.shape[2] == 1:
            x = x.squeeze(2)

        H = self.feature_extractor_part1(x)
        H = H.view(-1, 50 * 4 * 4)
        H = self.feature_extractor_part2(H)  # KxM

        A = self.attention(H)  # KxATTENTION_BRANCHES
        A = torch.transpose(A, 1, 0)  # ATTENTION_BRANCHESxK
        A = F.softmax(A, dim=1)  # softmax over K

        Z = torch.mm(A, H)  # ATTENTION_BRANCHESxM

        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()

        return Y_prob, Y_hat, A

    # AUXILIARY METHODS
    def calculate_classification_error(self, X, Y):
        Y = Y.float()
        _, Y_hat, _ = self.forward(X)
        error = 1. - Y_hat.eq(Y).cpu().float().mean().data.item()

        return error, Y_hat

    def calculate_objective(self, X, Y):
        Y = Y.float()
        Y_prob, _, A = self.forward(X)
        Y_prob = torch.clamp(Y_prob, min=1e-5, max=1. - 1e-5)
        neg_log_likelihood = -1. * (Y * torch.log(Y_prob) + (1. - Y) * torch.log(1. - Y_prob))  # negative log bernoulli

        return neg_log_likelihood, A

class GatedAttention(nn.Module):
    def __init__(self):
        super(GatedAttention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.feature_extractor_part1 = nn.Sequential(
            nn.Conv2d(1, 20, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(20, 50, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2)
        )

        self.feature_extractor_part2 = nn.Sequential(
            nn.Linear(50 * 4 * 4, self.M),
            nn.ReLU(),
        )

        self.attention_V = nn.Sequential(
            nn.Linear(self.M, self.L), # matrix V
            nn.Tanh()
        )

        self.attention_U = nn.Sequential(
            nn.Linear(self.M, self.L), # matrix U
            nn.Sigmoid()
        )

        self.attention_w = nn.Linear(self.L, self.ATTENTION_BRANCHES) # matrix w (or vector w if self.ATTENTION_BRANCHES==1)

        self.classifier = nn.Sequential(
            nn.Linear(self.M*self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.squeeze(0)
        # Add squeeze(2) to remove the unexpected singleton dimension if it exists
        if x.dim() == 5 and x.shape[2] == 1:
            x = x.squeeze(2)

        H = self.feature_extractor_part1(x)
        H = H.view(-1, 50 * 4 * 4)
        H = self.feature_extractor_part2(H)  # KxM

        A_V = self.attention_V(H)  # KxL
        A_U = self.attention_U(H)  # KxL
        A = self.attention_w(A_V * A_U) # element wise multiplication # KxATTENTION_BRANCHES
        A = torch.transpose(A, 1, 0)  # ATTENTION_BRANCHESxK
        A = F.softmax(A, dim=1)  # softmax over K

        Z = torch.mm(A, H)  # ATTENTION_BRANCHESxM

        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()

        return Y_prob, Y_hat, A

    # AUXILIARY METHODS
    def calculate_classification_error(self, X, Y):
        Y = Y.float()
        _, Y_hat, _ = self.forward(X)
        error = 1. - Y_hat.eq(Y).cpu().float().mean().item()

        return error, Y_hat

    def calculate_objective(self, X, Y):
        Y = Y.float()
        Y_prob, _, A = self.forward(X)
        Y_prob = torch.clamp(Y_prob, min=1e-5, max=1. - 1e-5)
        neg_log_likelihood = -1. * (Y * torch.log(Y_prob) + (1. - Y) * torch.log(1. - Y_prob))  # negative log bernoulli

        return neg_log_likelihood, A

In [ ]:
from __future__ import print_function

import argparse

import numpy as np
import torch
import torch.optim as optim
import torch.utils.data as data_utils
from torch.autograd import Variable


# Training settings
parser = argparse.ArgumentParser(description='PyTorch MNIST bags Example')
parser.add_argument('--epochs', type=int, default=20, metavar='N',
                    help='number of epochs to train (default: 20)')
parser.add_argument('--lr', type=float, default=0.0005, metavar='LR',
                    help='learning rate (default: 0.0005)')
parser.add_argument('--reg', type=float, default=10e-5, metavar='R',
                    help='weight decay')
parser.add_argument('--target_number', type=int, default=9, metavar='T',
                    help='bags have a positive labels if they contain at least one 9')
parser.add_argument('--mean_bag_length', type=int, default=10, metavar='ML',
                    help='average bag length')
parser.add_argument('--var_bag_length', type=int, default=2, metavar='VL',
                    help='variance of bag length')
parser.add_argument('--num_bags_train', type=int, default=200, metavar='NTrain',
                    help='number of bags in training set')
parser.add_argument('--num_bags_test', type=int, default=50, metavar='NTest',
                    help='number of bags in test set')
parser.add_argument('--seed', type=int, default=1, metavar='S',
                    help='random seed (default: 1)')
parser.add_argument('--no-cuda', action='store_true', default=False,
                    help='disables CUDA training')
parser.add_argument('--model', type=str, default='gated_attention', help='Choose b/w attention and gated_attention')

args = parser.parse_args([]) # Modified line: pass empty list to parse_args()
args.cuda = not args.no_cuda and torch.cuda.is_available()

torch.manual_seed(args.seed)
if args.cuda:
    torch.cuda.manual_seed(args.seed)
    print('\nGPU is ON!')

print('Load Train and Test Set')
loader_kwargs = {'num_workers': 1, 'pin_memory': True} if args.cuda else {}

train_loader = data_utils.DataLoader(MnistBags(target_number=args.target_number,
                                               mean_bag_length=args.mean_bag_length,
                                               var_bag_length=args.var_bag_length,
                                               num_bag=args.num_bags_train,
                                               seed=args.seed,
                                               train=True),
                                     batch_size=1,
                                     shuffle=True,
                                     **loader_kwargs)

test_loader = data_utils.DataLoader(MnistBags(target_number=args.target_number,
                                              mean_bag_length=args.mean_bag_length,
                                              var_bag_length=args.var_bag_length,
                                              num_bag=args.num_bags_test,
                                              seed=args.seed,
                                              train=False),
                                    batch_size=1,
                                    shuffle=False,
                                    **loader_kwargs)

print('Init Model')
if args.model=='attention':
    model = Attention()
elif args.model=='gated_attention':
    model = GatedAttention()
if args.cuda:
    model.cuda()

optimizer = optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.999), weight_decay=args.reg)


def train(epoch):
    model.train()
    train_loss = 0.
    train_error = 0.
    total, correct = 0, 0
    for batch_idx, (data, label) in enumerate(train_loader):
        bag_label = label[0]
        if args.cuda:
            data, bag_label = data.cuda(), bag_label.cuda()
        data, bag_label = Variable(data), Variable(bag_label)

        # reset gradients
        optimizer.zero_grad()
        # calculate loss and metrics
        loss, _ = model.calculate_objective(data, bag_label)
        train_loss += loss.data[0]
        error, predicted_bag_label = model.calculate_classification_error(data, bag_label) # Get Y_hat as predicted_bag_label
        train_error += error
        # backward pass
        loss.backward()
        # step
        optimizer.step()
        # Compare predicted_bag_label (float tensor) with bag_label (boolean tensor converted to float)
        correct += (predicted_bag_label == bag_label.float()).sum().item()
        total += 1

    # calculate loss and error for epoch
    train_loss /= len(train_loader)
    train_error /= len(train_loader)
    train_acc = correct / total

    print('Epoch: {}, Loss: {}, Train error: {}, train accuracy {}'.format(epoch, train_loss.cpu().numpy()[0], train_error, train_acc))


def test():
    model.eval()
    test_loss = 0.
    test_error = 0.
    total, correct = 0, 0
    with torch.no_grad():
        for batch_idx, (data, label) in enumerate(test_loader):
            bag_label = label[0]
            instance_labels = label[1]
            if args.cuda:
                data, bag_label = data.cuda(), bag_label.cuda()
            data, bag_label = Variable(data), Variable(bag_label)
            loss, attention_weights = model.calculate_objective(data, bag_label)
            test_loss += loss.data[0]
            error, predicted_bag_label = model.calculate_classification_error(data, bag_label) # Get Y_hat as predicted_bag_label
            test_error += error
            # Compare predicted_bag_label (float tensor) with bag_label (boolean tensor converted to float)
            correct += (predicted_bag_label == bag_label.float()).sum().item()
            total += 1

            if batch_idx < 5:  # plot bag labels and instance labels for first 5 bags
                bag_level = (bag_label.cpu().data.numpy()[0], int(predicted_bag_label.cpu().data.numpy()[0][0]))
                instance_level = list(zip(instance_labels.numpy()[0].tolist(),
                                    np.round(attention_weights.cpu().data.numpy()[0], decimals=3).tolist()))

                print('\nTrue Bag Label, Predicted Bag Label: {}\n'
                    'True Instance Labels, Attention Weights: {}'.format(bag_level, instance_level))

    test_error /= len(test_loader)
    test_loss /= len(test_loader)
    test_acc = correct / total

    print('\nTest Set, Loss: {}, Test error: {}, test accuracy {}'.format(test_loss.cpu().numpy()[0], test_error, test_acc))



print('Start Training')
for epoch in range(1, args.epochs + 1):
    train(epoch)
print('Start Testing')
test()


Load Train and Test Set
Init Model
Start Training
Epoch: 1, Loss: 0.742167055606842, Train error: 0.525, train accuracy 0.475
Epoch: 2, Loss: 0.6929910778999329, Train error: 0.5, train accuracy 0.5
Epoch: 3, Loss: 0.6880033016204834, Train error: 0.405, train accuracy 0.595
Epoch: 4, Loss: 0.6346940398216248, Train error: 0.33, train accuracy 0.67
Epoch: 5, Loss: 0.402833491563797, Train error: 0.165, train accuracy 0.835
Epoch: 6, Loss: 0.24266891181468964, Train error: 0.085, train accuracy 0.915
Epoch: 7, Loss: 0.2162005752325058, Train error: 0.085, train accuracy 0.915
Epoch: 8, Loss: 0.11721648275852203, Train error: 0.04, train accuracy 0.96
Epoch: 9, Loss: 0.04340367019176483, Train error: 0.01, train accuracy 0.99
Epoch: 10, Loss: 0.06065317243337631, Train error: 0.02, train accuracy 0.98
Epoch: 11, Loss: 0.05630310997366905, Train error: 0.015, train accuracy 0.985
Epoch: 12, Loss: 0.0030260540079325438, Train error: 0.0, train accuracy 1.0
Epoch: 13, Loss: 0.00147959275636

# Segmentación

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random
import matplotlib.pyplot as plt # Import matplotlib

# --- 1) Dataset MIL para "segmentación" sobre MNIST ---

def show_pairs(X, patches, numero, train=True, epsilon = None):
    X = np.array(X)

    n = len(patches)
    # Removed: idx = np.random.choice(X.shape[0], n, replace=False)
    plt.figure(figsize=(5 , 5))

    plt.imshow(X.squeeze(0), cmap='gray') # Squeeze channel dim for display
    plt.axis('off')
    plt.title(f'Patch {numero}')
    plt.show()
    fig, axes = plt.subplots(1, n, figsize=(1.6*n, 3.2))

    # Ensure axes is always a 2D array for consistent indexing.
    # If n=1, plt.subplots(2,1) returns a 1D array of shape (2). Actually, plt.subplots(1,1) returns a single axis object.
    # We want axes[row, col] consistently, so we reshape to (2,1).
    if n == 1:
        axes = np.array([[axes]]) # Wrap single axis in a 2D array for consistent indexing
    elif axes.ndim == 1 and n > 1:
        axes = np.expand_dims(axes, axis=0) # shape (1, n)

    for k, i_idx in enumerate(range(n)):
        axes[0, k].imshow(patches[i_idx].reshape(28, 28), cmap='gray')
        axes[0, k].axis('off')
        # axes[1, k].imshow(np.clip(Xhat[i_idx], 0, 1).reshape(28, 28), cmap='gray')
        # axes[1, k].axis('off')

    axes[0, 0].set_ylabel('original', rotation=90)
    # axes[1, 0].set_ylabel('recon', rotation=90)

    if train:
        plt.suptitle(f'MNIST 2 Train: σ2=1e-3, α=1.0, t=0.5, Markov=True, d = {128}')
    else:
        plt.suptitle(f'MNIST 2 Test: σ2=1e-3, α=1.0, t=0.5, Markov=True, d = {128}, ε={epsilon}')
    plt.tight_layout(); plt.show()

class MNISTSegBags(Dataset):
    """
    Cada ejemplo:
      - xs: tensor [n_patches, 1, 28, 28]  (instancias = parches reescalados)
      - y_bag: tensor [1]                  (OR de las etiquetas de parche)
      - y_inst: tensor [n_patches]         (0/1 por parche)
    """
    def __init__(self, train=True, grid_size=4, num_bags=10000, thr=0.1):
        self.ds = datasets.MNIST(
            root="./data",
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )
        self.grid_size = grid_size
        self.num_bags = num_bags
        self.thr = thr  # umbral para decidir si un parche contiene "dígito"

    def __len__(self):
        return self.num_bags

    def _split_into_patches(self, img, digit):
        """
        img: [1, 28, 28]
        Devuelve:
          patches: [n_patches, 1, 28, 28]
          inst_labels: [n_patches] en {0,1}
        """
        C, H, W = img.shape
        g = self.grid_size
        ph, pw = H // g, W // g   # tamaño del parche (e.g. 7x7)
        patches = []
        inst_labels = []

        for gy in range(g):
            for gx in range(g):
                patch = img[:, gy*ph:(gy+1)*ph, gx*pw:(gx+1)*pw]  # [1, ph, pw]
                # Etiqueta de parche: 1 si la intensidad media supera un umbral
                mean_val = patch.mean()
                label = 1.0 if mean_val > self.thr else 0.0

                # Reescalamos cada parche a 28x28 para usar el mismo encoder
                patch_resized = F.interpolate(
                    patch.unsqueeze(0),  # [1,1,ph,pw]
                    size=(28, 28),
                    mode="bilinear",
                    align_corners=False
                ).squeeze(0)  # [1,28,28]

                patches.append(patch_resized)
                inst_labels.append(label)

        patches = torch.stack(patches, dim=0)           # [n_patches,1,28,28]
        inst_labels = torch.tensor(inst_labels)         # [n_patches]
        return patches, inst_labels

    def __getitem__(self, _):
        # Elegimos una imagen MNIST al azar
        idx = random.randint(0, len(self.ds) - 1)
        img, digit = self.ds[idx]              # img: [1,28,28]

        xs, y_inst = self._split_into_patches(img, digit)  # [n_patches,1,28,28], [n_patches]

        show_pairs(img, xs, digit)

        # Etiqueta de bolsa = OR de las etiquetas de parche
        y_bag = y_inst.max().view(1)  # tensor [1] en {0,1}
        return xs, y_bag.float(), y_inst.float()

# --- 4) Entrenamiento para segmentación MIL (por parches) ---
def train_epoch_seg(model, train_loader, test_loader, opt,
                    device="cpu", lambda_bag=0.1):

    bce = nn.BCEWithLogitsLoss()

    # --- Fase de entrenamiento ---
    model.train()
    total_bags, correct_bags = 0, 0
    total_patches, correct_patches = 0, 0
    loss_sum = 0.0
    TP_tot = 0.0
    FP_tot = 0.0
    FN_tot = 0.0
    TN_tot = 0.0



    for idx,(xs, y_bag, y_inst) in enumerate(train_loader):



        xs = xs.to(device)[0]          # [n_patches,1,28,28]
        y_bag = y_bag.to(device)       # [1]
        y_inst = y_inst.to(device).squeeze(0)     # [n_patches] - FIX: Squeeze the batch dimension

        logit_bag, alpha, inst_logits = model(xs)

        # 1) Pérdida de "segmentación": instancia vs etiqueta de parche
        loss_inst = bce(inst_logits, y_inst)

        # 2) Pérdida de coherencia a nivel de bolsa:
        #    la etiqueta de bolsa es el OR de las etiquetas de parche
        target_bag = y_inst.max().view_as(y_bag)
        loss_bag = bce(logit_bag.view_as(target_bag), target_bag)

        loss = loss_inst + lambda_bag * loss_bag

        opt.zero_grad()
        loss.backward()
        opt.step()

        loss_sum += loss.item()

        # Métrica a nivel de bolsa (detección)
        pred_bag = (torch.sigmoid(logit_bag) >= 0.5).float()
        correct_bags += (pred_bag.view_as(target_bag) == target_bag).sum().item()
        total_bags += 1

        # Métrica a nivel de parche (segmentación binaria)
        pred_inst = (torch.sigmoid(inst_logits) >= 0.5).float()
        correct_patches += (pred_inst == y_inst).sum().item()
        total_patches += y_inst.numel()

        tp, fp, fn, tn = confusion_from_binary(pred_inst, y_inst)
        TP_tot += tp
        FP_tot += fp
        FN_tot += fn
        TN_tot += tn

        if idx < 10:
          # batch_size = 1 para simplificar; colapsamos el primer eje
          xs_display = xs.cpu() # Get the patches for display from the first bag in batch
          num_patches = xs_display.shape[0]

          plt.figure(figsize=(num_patches * 2, 2))
          for i in range(num_patches):
              plt.subplot(1, num_patches, i + 1)
              plt.imshow(xs_display[i].squeeze(0), cmap='gray') # Squeeze channel dim
              plt.axis('off')
              plt.title(f'Patch {i+1}')
              # FIX: Access the specific patch label from the batch tensor
          print(f"Label: {[y_inst[i].item() for i in range(num_patches)]}")
          print(f"Pred Label: {[pred_inst[i].item() for i in range(num_patches)]}")
          print(f"Label bag: {y_bag.item()}")
          plt.show()

    train_bag_acc = correct_bags / max(1, total_bags)
    train_patch_acc = correct_patches / max(1, total_patches)
    train_loss = loss_sum / max(1, total_bags)

    train_iou  = iou_score(TP_tot, FP_tot, FN_tot)
    train_dice = dice_score(TP_tot, FP_tot, FN_tot)
    train_prec = precision_score(TP_tot, FP_tot, FN_tot, TN_tot)
    train_rec  = recall_score(TP_tot, FP_tot, FN_tot, TN_tot)
    train_spec = specificity_score(TP_tot, FP_tot, FN_tot, TN_tot)
    train_bacc = balanced_accuracy(TP_tot, FP_tot, FN_tot, TN_tot)

    # --- Fase de evaluación ---
    model.eval()
    total_bags_t, correct_bags_t = 0, 0
    total_patches_t, correct_patches_t = 0, 0
    loss_sum_t = 0.0
    TP_tot = 0.0
    FP_tot = 0.0
    FN_tot = 0.0
    TN_tot = 0.0
    with torch.no_grad():
        for xs, y_bag, y_inst in test_loader:
            xs = xs.to(device)[0]
            y_bag = y_bag.to(device)
            y_inst = y_inst.to(device).squeeze(0) # FIX: Squeeze the batch dimension

            logit_bag, alpha, inst_logits = model(xs)

            loss_inst = bce(inst_logits, y_inst)
            target_bag = y_inst.max().view_as(y_bag)
            loss_bag = bce(logit_bag.view_as(target_bag), target_bag)
            loss = loss_inst + lambda_bag * loss_bag

            loss_sum_t += loss.item()

            pred_bag = (torch.sigmoid(logit_bag) >= 0.5).float()
            correct_bags_t += (pred_bag.view_as(target_bag) == target_bag).sum().item()
            total_bags_t += 1

            pred_inst = (torch.sigmoid(inst_logits) >= 0.5).float()
            correct_patches_t += (pred_inst == y_inst).sum().item()
            total_patches_t += y_inst.numel()

            tp, fp, fn, tn = confusion_from_binary(pred_inst, y_inst)
            TP_tot += tp
            FP_tot += fp
            FN_tot += fn
            TN_tot += tn

    test_bag_acc = correct_bags_t / max(1, total_bags_t)
    test_patch_acc = correct_patches_t / max(1, total_patches_t)
    test_loss = loss_sum_t / max(1, total_bags_t)

    # Métricas de segmentación agregadas
    test_iou  = iou_score(TP_tot, FP_tot, FN_tot)
    test_dice = dice_score(TP_tot, FP_tot, FN_tot)
    test_prec = precision_score(TP_tot, FP_tot, FN_tot, TN_tot)
    test_rec  = recall_score(TP_tot, FP_tot, FN_tot, TN_tot)
    test_spec = specificity_score(TP_tot, FP_tot, FN_tot, TN_tot)
    test_bacc = balanced_accuracy(TP_tot, FP_tot, FN_tot, TN_tot)

    metrics = {
        "train_loss": train_loss,
        "train_bag_acc": train_bag_acc,
        "train_inst_acc": train_patch_acc,
        "train_IoU": train_iou,
        "train_Dice": train_dice,
        "train_Precision": train_prec,
        "train_Recall": train_rec,
        "train_Specificity": train_spec,
        "train_BalancedAcc": train_bacc,
        "test_loss": test_loss,
        "test_bag_acc": test_bag_acc,
        "test_inst_acc": test_patch_acc,
        "test_IoU": test_iou,
        "test_Dice": test_dice,
        "test_Precision": test_prec,
        "test_Recall": test_rec,
        "test_Specificity": test_spec,
        "test_BalancedAcc": test_bacc
    }

    return metrics

# Uso:
train_loader = DataLoader(
    MNISTSegBags(train=True, grid_size=4, num_bags=2000, thr=0.1),
    batch_size=1,
    shuffle=True
)
test_loader = DataLoader(
    MNISTSegBags(train=False, grid_size=4, num_bags=500, thr=0.1),
    batch_size=1
)

device = "cpu"
model = AttentionMIL(d=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    metrics = train_epoch_seg(model, train_loader, test_loader, opt,
                        device=device, lambda_bag=0.1)

    print(f"Epoch {epoch}: "
          f"train_loss={metrics['train_loss']:.3f}  "
          f"train_bag_acc={metrics['train_bag_acc']:.3f}  "
          f"train_inst_acc={metrics['train_inst_acc']:.3f}  "
          f"train_IoU={metrics['train_IoU']:.3f}  "
          f"train_Dice={metrics['train_Dice']:.3f}  |  "
          f"test_loss={metrics['test_loss']:.3f}  "
          f"test_bag_acc={metrics['test_bag_acc']:.3f}  "
          f"test_inst_acc={metrics['test_inst_acc']:.3f}  "
          f"test_IoU={metrics['test_IoU']:.3f}  "
          f"test_Dice={metrics['test_Dice']:.3f}")

In [ ]:
train_loader = DataLoader(
    MNISTSegBags(train=True, grid_size=4, num_bags=2000, thr=0.1),
    batch_size=1,
    shuffle=True
)
test_loader = DataLoader(
    MNISTSegBags(train=False, grid_size=4, num_bags=500, thr=0.1),
    batch_size=1
)

device = "cpu"
model = AttentionMIL(d=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    metrics = train_epoch_seg(model, train_loader, test_loader, opt,
                        device=device, lambda_bag=0.0)

    print(f"Epoch {epoch}: "
          f"train_loss={metrics['train_loss']:.3f}  "
          f"train_bag_acc={metrics['train_bag_acc']:.3f}  "
          f"train_inst_acc={metrics['train_inst_acc']:.3f}  "
          f"train_IoU={metrics['train_IoU']:.3f}  "
          f"train_Dice={metrics['train_Dice']:.3f}  |  "
          f"test_loss={metrics['test_loss']:.3f}  "
          f"test_bag_acc={metrics['test_bag_acc']:.3f}  "
          f"test_inst_acc={metrics['test_inst_acc']:.3f}  "
          f"test_IoU={metrics['test_IoU']:.3f}  "
          f"test_Dice={metrics['test_Dice']:.3f}")

In [ ]:
import torch
from torchvision import datasets, transforms

class MNISTSegBags(object):
    """
    Cada elemento:
        X_bag : (n_patches, d)   – features de parches (flatten)
        y_bag : escalar 0/1      – etiqueta de bolsa (OR de parches positivos)
        y_inst: (n_patches,)     – etiquetas 0/1 por parche
    """
    def __init__(self, train=True, num_bags=400, grid_size=4, thr=0.1):
        self.grid_size = grid_size
        self.num_bags = num_bags
        self.thr = thr

        self.ds = datasets.MNIST(
            root="./data",
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )

    def __len__(self):
        return self.num_bags

    def _image_to_patches(self, img):
        """
        img: tensor [1,28,28]
        Devuelve:
            X_patches: (n_patches, d)
            y_inst   : (n_patches,)
        """
        img_np = img.numpy()  # [1,28,28]
        _, H, W = img_np.shape
        g = self.grid_size
        ph, pw = H // g, W // g  # tamaño de parche, p.ej. 7x7

        patches = []
        labels = []

        for gy in range(g):
            for gx in range(g):
                patch = img_np[:, gy*ph:(gy+1)*ph, gx*pw:(gx+1)*pw]  # [1,ph,pw]
                mean_val = patch.mean()
                label = 1.0 if mean_val > self.thr else 0.0

                patches.append(patch.reshape(-1))  # flatten
                labels.append(label)

        X_patches = np.stack(patches, axis=0).astype(np.float32)  # (n_patches, d)
        y_inst = np.array(labels, dtype=np.float32)               # (n_patches,)
        return X_patches, y_inst

    def __getitem__(self, idx):
        # elegimos una imagen aleatoria de MNIST
        r_idx = random.randint(0, len(self.ds) - 1)
        img, digit = self.ds[r_idx]  # img: [1,28,28]

        X_patches, y_inst = self._image_to_patches(img)
        y_bag = float(y_inst.max())  # OR de labels de parches

        return X_patches, y_bag, y_inst

def convertToBatch_seg(list_of_bags):
    """
    list_of_bags: lista de triples (X_bag, y_bag, y_inst)
    Devuelve lista de batches:
        [X, y_bag_array, y_inst_array]
        X           : (n_patches, d)
        y_bag_array : (1,1)
        y_inst_array: (n_patches,1)
    """
    batch_list = []
    for (X_bag, y_bag, y_inst) in list_of_bags:
        X = np.asarray(X_bag, dtype=np.float32)                     # (n_patches, d)
        yb = np.array([[y_bag]], dtype=np.float32)                  # (1,1)
        yi = np.asarray(y_inst, dtype=np.float32).reshape(-1, 1)    # (n_patches,1)
        batch_list.append([X, yb, yi])
    return batch_list

def load_dataset_seg(dataset_nm, n_folds, num_bags=400, grid_size=4, thr=0.1):
    """
    Construye un toy dataset de segmentación sobre MNIST y lo divide en k-fold.
    """
    mnist_dataset = MNISTSegBags(train=True,
                                 num_bags=num_bags,
                                 grid_size=grid_size,
                                 thr=thr)
    all_bags = [mnist_dataset[i] for i in range(len(mnist_dataset))]
    num_bag = len(all_bags)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=None)
    datasets = []
    for train_idx, test_idx in kf.split(range(num_bag)):
        dataset = {}
        dataset['train'] = [all_bags[ibag] for ibag in train_idx]
        dataset['test']  = [all_bags[ibag] for ibag in test_idx]
        datasets.append(dataset)
    return datasets


def train_eval_seg(model, train_set):
    """
    Bucle de entrenamiento por época con métricas de segmentación.
    Devuelve un diccionario con loss, acc de bolsa y métricas de parches.
    """
    num_batch = len(train_set)
    loss_arr = np.zeros((num_batch, 1), dtype=float)
    bag_acc_arr = np.zeros((num_batch, 1), dtype=float)
    inst_acc_arr = np.zeros((num_batch, 1), dtype=float)

    TP_tot, FP_tot, FN_tot, TN_tot = 0, 0, 0, 0

    shuffle(train_set)
    # Get grid_size from the dataset to correctly reshape patches
    # Assuming grid_size is fixed for the dataset and can be derived from num_bags and flattened patch dim
    # Or, preferably, pass it as an argument or retrieve from a global/class variable if available
    # For this specific case, we know grid_size=4 and MNIST image size=28x28, so patch_dim = 28/4 = 7
    patch_dim = 28 // MNISTSegBags.grid_size if hasattr(MNISTSegBags, 'grid_size') else 7 # Default to 7 if not set

    for idx, batch in enumerate(train_set):
      if idx < 10:
        xs_bag, y_bag_np, y_inst_np = batch # Unpack the batch

        num_patches = xs_bag.shape[0]

        plt.figure(figsize=(num_patches * 2, 2))
        for i in range(num_patches):
            patch_data = xs_bag[i].reshape(patch_dim, patch_dim) # Reshape flattened patch to 2D image (7x7)
            plt.subplot(1, num_patches, i + 1)
            plt.imshow(patch_data, cmap='gray')
            plt.axis('off')
            plt.title(f'Patch {i+1}')
        print(f"Label: {[y_inst_np[i, 0].item() for i in range(num_patches)]}")
        print(f"Label bag: {y_bag_np.item()}")
        plt.show()

    for ibatch, batch in enumerate(train_set):
        X, y_bag, y_inst = batch   # y_inst: (n_patches,1)

        # 1) paso de entrenamiento
        result = model.train_on_batch({'input': X}, [y_bag, y_inst])
        # train_on_batch devuelve [total_loss, loss_bag, loss_inst, ...] si hay más outputs
        if isinstance(result, (list, tuple)):
            loss_total = result[0]
        else:
            loss_total = result
        loss_arr[ibatch] = loss_total

        # 2) predicciones para métricas
        y_bag_hat, y_inst_hat = model.predict_on_batch({'input': X})
        # y_bag_hat: (1,1), y_inst_hat: (n_patches,1)

        # -- bolsa --
        bag_pred = (y_bag_hat >= 0.5).astype(float)
        bag_acc = (bag_pred == y_bag).mean()
        bag_acc_arr[ibatch] = bag_acc

        # -- parches --
        inst_pred = (y_inst_hat >= 0.5).astype(float)
        inst_true = y_inst

        inst_acc = (inst_pred == inst_true).mean()
        inst_acc_arr[ibatch] = inst_acc

        inst_pred_flat = inst_pred.reshape(-1)
        inst_true_flat = inst_true.reshape(-1)

        tp, fp, fn, tn = confusion_from_binary(inst_true_flat, inst_pred_flat)
        TP_tot += tp
        FP_tot += fp
        FN_tot += fn
        TN_tot += tn

    mean_loss_total = np.mean(loss_arr)
    mean_bag_acc = np.mean(bag_acc_arr)
    mean_inst_acc = np.mean(inst_acc_arr)

    iou  = iou_score(TP_tot, FP_tot, FN_tot)
    dice = dice_score(TP_tot, FP_tot, FN_tot)
    prec = precision_score(TP_tot, FP_tot, FN_tot, TN_tot)
    rec  = recall_score(TP_tot, FP_tot, FN_tot, TN_tot)
    spec = specificity_score(TP_tot, FP_tot, FN_tot, TN_tot)
    bacc = balanced_accuracy(TP_tot, FP_tot, FN_tot, TN_tot)

    metrics = {
        "loss": mean_loss_total,
        "bag_acc": mean_bag_acc,
        "inst_acc": mean_inst_acc,
        "IoU": iou,
        "Dice": dice,
        "Precision": prec,
        "Recall": rec,
        "Specificity": spec,
        "BalancedAcc": bacc
    }
    return metrics

def test_eval_seg(model, test_set):
    """
    Bucle de evaluación con métricas de segmentación.
    """
    num_batch = len(test_set)
    loss_arr = np.zeros((num_batch, 1), dtype=float)
    bag_acc_arr = np.zeros((num_batch, 1), dtype=float)
    inst_acc_arr = np.zeros((num_batch, 1), dtype=float)

    TP_tot, FP_tot, FN_tot, TN_tot = 0, 0, 0, 0

    for ibatch, batch in enumerate(test_set):
        X, y_bag, y_inst = batch

        result = model.test_on_batch({'input': X}, [y_bag, y_inst])
        if isinstance(result, (list, tuple)):
            loss_total = result[0]
        else:
            loss_total = result
        loss_arr[ibatch] = loss_total

        y_bag_hat, y_inst_hat = model.predict_on_batch({'input': X})

        bag_pred = (y_bag_hat >= 0.5).astype(float)
        bag_acc = (bag_pred == y_bag).mean()
        bag_acc_arr[ibatch] = bag_acc

        inst_pred = (y_inst_hat >= 0.5).astype(float)
        inst_true = y_inst

        inst_acc = (inst_pred == inst_true).mean()
        inst_acc_arr[ibatch] = inst_acc

        inst_pred_flat = inst_pred.reshape(-1)
        inst_true_flat = inst_true.reshape(-1)

        tp, fp, fn, tn = confusion_from_binary(inst_true_flat, inst_pred_flat)
        TP_tot += tp
        FP_tot += fp
        FN_tot += fn
        TN_tot += tn

    mean_loss_total = np.mean(loss_arr)
    mean_bag_acc = np.mean(bag_acc_arr)
    mean_inst_acc = np.mean(inst_acc_arr)

    iou  = iou_score(TP_tot, FP_tot, FN_tot)
    dice = dice_score(TP_tot, FP_tot, FN_tot)
    prec = precision_score(TP_tot, FP_tot, FN_tot, TN_tot)
    rec  = recall_score(TP_tot, FP_tot, FN_tot, TN_tot)
    spec = specificity_score(TP_tot, FP_tot, FN_tot, TN_tot)
    bacc = balanced_accuracy(TP_tot, FP_tot, FN_tot, TN_tot)

    metrics = {
        "loss": mean_loss_total,
        "bag_acc": mean_bag_acc,
        "inst_acc": mean_inst_acc,
        "IoU": iou,
        "Dice": dice,
        "Precision": prec,
        "Recall": rec,
        "Specificity": spec,
        "BalancedAcc": bacc
    }
    return metrics


def MI_Net_seg(dataset, args):
    """
    Versión de MI-Net para segmentación por parches.
    """
    # 1) datos
    train_bags = dataset['train']
    test_bags  = dataset['test']

    train_set = convertToBatch_seg(train_bags)
    test_set  = convertToBatch_seg(test_bags)

    dimension = train_set[0][0].shape[1]  # d = dimensión de parche flatten

    # 2) modelo MIL
    data_input = Input(shape=(dimension,), dtype='float32', name='input')

    fc1 = Dense(256, activation='relu', kernel_regularizer=l2(args.weight_decay))(data_input)
    fc2 = Dense(128, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc1)
    fc3 = Dense(64, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc2)

    dropout = Dropout(rate=0.5)(fc3)

    # salida instancia (segmentación)
    inst_out = Dense(1,
                     activation='sigmoid',
                     kernel_regularizer=l2(args.weight_decay),
                     name='inst_out')(dropout)  # (n_patches,1)

    # salida bolsa (MIL pooling sobre features)
    fp = Feature_pooling(output_dim=1,
                         kernel_regularizer=l2(args.weight_decay),
                         pooling_mode=args.pooling_mode,
                         name='bag_out')(dropout)  # (1,)

    model = Model(inputs=[data_input], outputs=[fp, inst_out])

    sgd = SGD(learning_rate=args.init_lr,
              momentum=args.momentum,
              nesterov=True)

    # Dos pérdidas: bolsa + instancia
    # bag_loss es tu función MIL (por ejemplo, BCE sobre labels de bolsa)
    model.compile(
        loss=[bag_loss, 'binary_crossentropy'],
        loss_weights=[1.0, 1.0],   # puedes ajustar el peso de la instancia
        optimizer=sgd
    )

    # 3) entrenamiento
    t1 = time.time()
    for epoch in range(args.max_epoch):
        train_metrics = train_eval_seg(model, train_set)
        test_metrics  = test_eval_seg(model, test_set)

        print(
            f"epoch={epoch:03d}  "
            f"train_loss={train_metrics['loss']:.3f}  "
            f"train_bag_acc={train_metrics['bag_acc']:.3f}  "
            f"train_inst_acc={train_metrics['inst_acc']:.3f}  "
            f"train_IoU={train_metrics['IoU']:.3f}  "
            f"train_Dice={train_metrics['Dice']:.3f}  |  "
            f"test_loss={test_metrics['loss']:.3f}  "
            f"test_bag_acc={test_metrics['bag_acc']:.3f}  "
            f"test_inst_acc={test_metrics['inst_acc']:.3f}  "
            f"test_IoU={test_metrics['IoU']:.3f}  "
            f"test_Dice={test_metrics['Dice']:.3f}"
        )

    t2 = time.time()
    print('run time:', (t2 - t1) / 60.0, 'min')
    print('final test_bag_acc={:.3f}'.format(test_metrics['bag_acc']))
    print('final test_inst_acc={:.3f}'.format(test_metrics['inst_acc']))

    return test_metrics

args = parse_args()
print('Called with args:')
print(args)

run = 1
n_folds = 3
bag_acc_all = np.zeros((run, n_folds), dtype=float)
inst_acc_all = np.zeros((run, n_folds), dtype=float)

for irun in range(run):
    datasets = load_dataset_seg(args.dataset, n_folds,
                                num_bags=400, grid_size=4, thr=0.1)

    for ifold in range(n_folds):
        print('run=', irun, '  fold=', ifold)
        metrics = MI_Net_seg(datasets[ifold], args)
        bag_acc_all[irun][ifold]  = metrics['bag_acc']
        inst_acc_all[irun][ifold] = metrics['inst_acc']

print('MI-Net (seg) mean bag accuracy = ', np.mean(bag_acc_all))
print('bag std = ', np.std(bag_acc_all))
print('MI-Net (seg) mean inst accuracy = ', np.mean(inst_acc_all))
print('inst std = ', np.std(inst_acc_all))

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import torch
import numpy as np
import random # Add this import

class MNISTSegBags(object):
    """
    Cada elemento:
        X_bag : (n_patches, d)   – features de parches (flatten)
        y_bag : escalar 0/1      – etiqueta de bolsa (OR de parches positivos)
        y_inst: (n_patches,)     – etiquetas 0/1 por parche
    """
    def __init__(self, train=True, num_bags=400, grid_size=4, thr=0.1):
        self.grid_size = grid_size
        self.num_bags = num_bags
        self.thr = thr

        self.ds = datasets.MNIST(
            root="./data",
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )

    def __len__(self):
        return self.num_bags

    def _image_to_patches(self, img):
        """
        img: tensor [1,28,28]
        Devuelve:
            X_patches: (n_patches, d)
            y_inst   : (n_patches,)
        """
        img_np = img.numpy()  # [1,28,28]
        _, H, W = img_np.shape
        g = self.grid_size
        ph, pw = H // g, W // g  # tamaño de parche, p.ej. 7x7

        patches = []
        labels = []

        for gy in range(g):
            for gx in range(g):
                patch = img_np[:, gy*ph:(gy+1)*ph, gx*pw:(gx+1)*pw]  # [1,ph,pw]
                mean_val = patch.mean()
                label = 1.0 if mean_val > self.thr else 0.0

                patches.append(patch.reshape(-1))  # flatten
                labels.append(label)

        X_patches = np.stack(patches, axis=0).astype(np.float32)  # (n_patches, d)
        y_inst = np.array(labels, dtype=np.float32)               # (n_patches,)
        return X_patches, y_inst

    def __getitem__(self, idx):
        # elegimos una imagen aleatoria de MNIST
        r_idx = random.randint(0, len(self.ds) - 1)
        img, digit = self.ds[r_idx]  # img: [1,28,28]

        X_patches, y_inst = self._image_to_patches(img)
        y_bag = float(y_inst.max())  # OR de labels de parches

        return X_patches, y_bag, y_inst


# ----------------------------------------------------------------------
# 4) Conversión de bolsas a "batches" y k-fold
# ----------------------------------------------------------------------
def convertToBatch(list_of_bags):
    """
    list_of_bags: lista de triples (X_bag, y_bag, y_inst)
    Devuelve lista de batches:
        [X, y_bag_array, y_inst_array]
        X           : (n_patches, d)
        y_bag_array : (1,1)
        y_inst_array: (n_patches,1)
    """
    batch_list = []
    for (X_bag, y_bag, y_inst) in list_of_bags:
        X = np.asarray(X_bag, dtype=np.float32)                     # (n_patches, d)
        yb = np.array([[y_bag]], dtype=np.float32)                  # (1,1)
        yi = np.asarray(y_inst, dtype=np.float32).reshape(-1, 1)    # (n_patches,1)
        batch_list.append([X, yb, yi])
    return batch_list


def load_dataset(dataset_nm, n_folds, num_bags=400, grid_size=4, thr=0.1):
    """
    Construye un toy dataset de segmentación sobre MNIST y lo divide en k-fold.
    """
    mnist_dataset = MNISTSegBags(train=True,
                                 num_bags=num_bags,
                                 grid_size=grid_size,
                                 thr=thr)
    all_bags = [mnist_dataset[i] for i in range(len(mnist_dataset))]
    num_bag = len(all_bags)

    from sklearn.model_selection import KFold
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=None)
    datasets = []
    for train_idx, test_idx in kf.split(range(num_bag)):
        dataset = {}
        dataset['train'] = [all_bags[ibag] for ibag in train_idx]
        dataset['test']  = [all_bags[ibag] for ibag in test_idx]
        datasets.append(dataset)
    return datasets


# ----------------------------------------------------------------------
# 5) train_step y test_step con métricas de segmentación
# ----------------------------------------------------------------------
def train_eval(model, train_set):
    """
    Bucle de entrenamiento por época con métricas de segmentación.
    Devuelve un diccionario con loss, acc de bolsa y métricas de parches.
    """
    num_train_batch = len(train_set)
    loss_arr = np.zeros((num_train_batch, 1), dtype=float)
    bag_acc_arr = np.zeros((num_train_batch, 1), dtype=float)
    inst_acc_arr = np.zeros((num_train_batch, 1), dtype=float)

    TP_tot, FP_tot, FN_tot, TN_tot = 0, 0, 0, 0

    shuffle(train_set)
    patch_dim = 28 // MNISTSegBags.grid_size if hasattr(MNISTSegBags, 'grid_size') else 7 # Default to 7 if not set

    for idx, batch in enumerate(train_set):
      if idx < 10:
        xs_bag, y_bag_np, y_inst_np = batch # Unpack the batch

        num_patches = xs_bag.shape[0]

        plt.figure(figsize=(num_patches * 2, 2))
        for i in range(num_patches):
            patch_data = xs_bag[i].reshape(patch_dim, patch_dim) # Reshape flattened patch to 2D image (7x7)
            plt.subplot(1, num_patches, i + 1)
            plt.imshow(patch_data, cmap='gray')
            plt.axis('off')
            plt.title(f'Patch {i+1}')
        print(f"Label: {[y_inst_np[i, 0].item() for i in range(num_patches)]}")
        print(f"Label bag: {y_bag_np.item()}")
        plt.show()

    for ibatch, batch in enumerate(train_set):
        X, y_bag, y_inst = batch   # y_inst: (n_patches,1)

        # 1) paso de entrenamiento
        result = model.train_on_batch({'input': X}, [y_bag, y_inst])
        # Keras devuelve [total_loss, loss_bag, loss_inst] si hay dos salidas
        if isinstance(result, (list, tuple)):
            loss_total = result[0]
        else:
            loss_total = result
        loss_arr[ibatch] = loss_total

        # 2) predicciones para métricas
        y_bag_hat, y_inst_hat = model.predict_on_batch({'input': X})
        # y_bag_hat: (1,1), y_inst_hat: (n_patches,1)

        # -- bolsa --
        bag_pred = (y_bag_hat >= 0.5).astype(float)
        bag_acc = (bag_pred == y_bag).mean()
        bag_acc_arr[ibatch] = bag_acc

        # -- parches --
        inst_pred = (y_inst_hat >= 0.5).astype(float)
        inst_true = y_inst

        inst_acc = (inst_pred == inst_true).mean()
        inst_acc_arr[ibatch] = inst_acc

        inst_pred_flat = inst_pred.reshape(-1)
        inst_true_flat = inst_true.reshape(-1)

        tp, fp, fn, tn = confusion_from_binary(inst_true_flat, inst_pred_flat)
        TP_tot += tp
        FP_tot += fp
        FN_tot += fn
        TN_tot += tn

    mean_loss_total = np.mean(loss_arr)
    mean_bag_acc = np.mean(bag_acc_arr)
    mean_inst_acc = np.mean(inst_acc_arr)

    iou  = iou_score(TP_tot, FP_tot, FN_tot)
    dice = dice_score(TP_tot, FP_tot, FN_tot)
    prec = precision_score(TP_tot, FP_tot, FN_tot, TN_tot)
    rec  = recall_score(TP_tot, FP_tot, FN_tot, TN_tot)
    spec = specificity_score(TP_tot, FP_tot, FN_tot, TN_tot)
    bacc = balanced_accuracy(TP_tot, FP_tot, FN_tot, TN_tot)

    metrics = {
        "loss": mean_loss_total,
        "bag_acc": mean_bag_acc,
        "inst_acc": mean_inst_acc,
        "IoU": iou,
        "Dice": dice,
        "Precision": prec,
        "Recall": rec,
        "Specificity": spec,
        "BalancedAcc": bacc
    }
    return metrics


def test_eval(model, test_set):
    """
    Bucle de evaluación con métricas de segmentación.
    """
    num_test_batch = len(test_set)
    test_loss = np.zeros((num_test_batch, 1), dtype=float)
    bag_acc_arr = np.zeros((num_test_batch, 1), dtype=float)
    inst_acc_arr = np.zeros((num_test_batch, 1), dtype=float)

    TP_tot, FP_tot, FN_tot, TN_tot = 0, 0, 0, 0

    for ibatch, batch in enumerate(test_set):
        X, y_bag, y_inst = batch

        result = model.test_on_batch({'input': X}, [y_bag, y_inst])
        if isinstance(result, (list, tuple)):
            loss_total = result[0]
        else:
            loss_total = result
        test_loss[ibatch] = loss_total

        y_bag_hat, y_inst_hat = model.predict_on_batch({'input': X})

        bag_pred = (y_bag_hat >= 0.5).astype(float)
        bag_acc = (bag_pred == y_bag).mean()
        bag_acc_arr[ibatch] = bag_acc

        inst_pred = (y_inst_hat >= 0.5).astype(float)
        inst_true = y_inst

        inst_acc = (inst_pred == inst_true).mean()
        inst_acc_arr[ibatch] = inst_acc

        inst_pred_flat = inst_pred.reshape(-1)
        inst_true_flat = inst_true.reshape(-1)

        tp, fp, fn, tn = confusion_from_binary(inst_true_flat, inst_pred_flat)
        TP_tot += tp
        FP_tot += fp
        FN_tot += fn
        TN_tot += tn

    mean_loss_total = np.mean(test_loss)
    mean_bag_acc = np.mean(bag_acc_arr)
    mean_inst_acc = np.mean(inst_acc_arr)

    iou  = iou_score(TP_tot, FP_tot, FN_tot)
    dice = dice_score(TP_tot, FP_tot, FN_tot)
    prec = precision_score(TP_tot, FP_tot, FN_tot, TN_tot)
    rec  = recall_score(TP_tot, FP_tot, FN_tot, TN_tot)
    spec = specificity_score(TP_tot, FP_tot, FN_tot, TN_tot)
    bacc = balanced_accuracy(TP_tot, FP_tot, FN_tot, TN_tot)

    metrics = {
        "loss": mean_loss_total,
        "bag_acc": mean_bag_acc,
        "inst_acc": mean_inst_acc,
        "IoU": iou,
        "Dice": dice,
        "Precision": prec,
        "Recall": rec,
        "Specificity": spec,
        "BalancedAcc": bacc
    }
    return metrics


# ----------------------------------------------------------------------
# 6) mi-Net para segmentación: dos salidas (bolsa e instancia)
# ----------------------------------------------------------------------
def mi_Net(dataset):
    """
    Versión de mi-Net para segmentación por parches.
    """
    # datos
    train_bags = dataset['train']
    test_bags  = dataset['test']

    train_set = convertToBatch(train_bags)
    test_set  = convertToBatch(test_bags)
    dimension = train_set[0][0].shape[1]  # d = dimensión de parche flatten

    # entrada de instancia
    data_input = Input(shape=(dimension,), dtype='float32', name='input')

    # fully-connected backbone
    fc1 = Dense(256, activation='relu', kernel_regularizer=l2(args.weight_decay))(data_input)
    fc2 = Dense(128, activation='relu', kernel_regularizer=l2(args.weight_decay))(fc1)
    fc3 = Dense(64,  activation='relu', kernel_regularizer=l2(args.weight_decay))(fc2)

    dropout = Dropout(rate=0.5)(fc3)

    # salida instancia (segmentación por parche)
    inst_out = Dense(
        1,
        activation='sigmoid',
        kernel_regularizer=l2(args.weight_decay),
        name='inst_out'
    )(dropout)  # (n_patches,1)

    # salida de bolsa vía Score_pooling
    sp = Score_pooling(
        output_dim=1,
        kernel_regularizer=l2(args.weight_decay),
        pooling_mode=args.pooling_mode,
        name='sp'
    )(dropout)  # (1,)

    model = Model(inputs=[data_input], outputs=[sp, inst_out])

    sgd = SGD(learning_rate=args.init_lr,
              momentum=args.momentum,
              nesterov=True)

    # Dos pérdidas: bolsa + instancia
    # bag_loss: tu función MIL de bolsa (por ejemplo BCE de bolsa)
    model.compile(
        loss=[bag_loss, 'binary_crossentropy'],
        loss_weights=[1.0, 1.0],
        optimizer=sgd
    )

    # entrenamiento
    t1 = time.time()
    for epoch in range(args.max_epoch):
        train_metrics = train_eval(model, train_set)
        test_metrics  = test_eval(model, test_set)

        print(
            f"epoch={epoch:03d}  "
            f"train_loss={train_metrics['loss']:.3f}  "
            f"train_bag_acc={train_metrics['bag_acc']:.3f}  "
            f"train_inst_acc={train_metrics['inst_acc']:.3f}  "
            f"train_IoU={train_metrics['IoU']:.3f}  "
            f"train_Dice={train_metrics['Dice']:.3f}  |  "
            f"test_loss={test_metrics['loss']:.3f}  "
            f"test_bag_acc={test_metrics['bag_acc']:.3f}  "
            f"test_inst_acc={test_metrics['inst_acc']:.3f}  "
            f"test_IoU={test_metrics['IoU']:.3f}  "
            f"test_Dice={test_metrics['Dice']:.3f}"
        )

    t2 = time.time()
    print('run time:', (t2 - t1) / 60.0, 'min')
    print('final test_bag_acc={:.3f}'.format(test_metrics['bag_acc']))
    print('final test_inst_acc={:.3f}'.format(test_metrics['inst_acc']))

    # Devolvemos accuracy de bolsa (para el promedio de folds)
    return test_metrics['bag_acc']


# ----------------------------------------------------------------------
# 7) Bucle principal k-fold
# ----------------------------------------------------------------------
args = parse_args()
print('Called with args:')
print(args)

run = 1
n_folds = 3
acc = np.zeros((run, n_folds), dtype=float)

for irun in range(run):
    datasets = load_dataset(args.dataset, n_folds,
                            num_bags=400, grid_size=4, thr=0.1)
    for ifold in range(n_folds):
        print('run=', irun, '  fold=', ifold)
        acc[irun][ifold] = mi_Net(datasets[ifold])

print('mi-net (seg) mean bag accuracy = ', np.mean(acc))
print('std = ', np.std(acc))

In [ ]:
import torch
import torch.nn.functional as F
import torch.utils.data as data_utils
from torchvision import datasets, transforms
import numpy as np # Added numpy import

class MnistSegBags(data_utils.Dataset):
    """
    Cada bag = 1 imagen MNIST.
    Cada instancia = 1 parche en una rejilla g x g (K = g^2).
    Retorna:
      bag:   [K, 1, 28, 28]   (parches reescalados)
      label: [bag_label, inst_labels]
             bag_label: tensor [1] (OR de inst_labels)
             inst_labels: tensor [K] (0/1 por parche)
    """
    def __init__(self, grid_size=4, num_bag=1000, seed=7, train=True, thr=0.10):
        self.grid_size = grid_size
        self.num_bag = num_bag
        self.r = np.random.RandomState(seed)
        self.train = train
        self.thr = thr

        self.ds = datasets.MNIST(
            '../datasets',
            train=train,
            download=True,
            transform=transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.1307,), (0.3081,))
            ])
        )
        self.N = len(self.ds)

    def __len__(self):
        return self.num_bag

    def _to_patches(self, img):
        # img: [1,28,28]
        g = self.grid_size
        _, H, W = img.shape
        ph, pw = H // g, W // g

        patches = []
        inst_labels = []

        for gy in range(g):
            for gx in range(g):
                p = img[:, gy*ph:(gy+1)*ph, gx*pw:(gx+1)*pw]  # [1,ph,pw]
                # etiqueta del parche: “contiene dígito” si media > umbral
                lab = 1.0 if p.mean().item() > self.thr else 0.0

                # reescalar parche a 28x28 para compatibilidad con tu CNN
                p28 = F.interpolate(
                    p.unsqueeze(0), size=(28, 28),
                    mode="bilinear", align_corners=False
                ).squeeze(0)  # [1,28,28]

                patches.append(p28)
                inst_labels.append(lab)

        bag = torch.stack(patches, dim=0)                       # [K,1,28,28]
        inst_labels = torch.tensor(inst_labels, dtype=torch.float32)  # [K]
        bag_label = inst_labels.max().view(1)                   # [1]
        return bag, bag_label, inst_labels

    def __getitem__(self, index):
        idx = int(self.r.randint(0, self.N))
        img, _ = self.ds[idx]
        bag, bag_label, inst_labels = self._to_patches(img)
        return bag, [bag_label, inst_labels]

import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.feature_extractor_part1 = nn.Sequential(
            nn.Conv2d(1, 20, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(20, 50, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2)
        )

        self.feature_extractor_part2 = nn.Sequential(
            nn.Linear(50 * 4 * 4, self.M),
            nn.ReLU(),
        )

        self.attention = nn.Sequential(
            nn.Linear(self.M, self.L),
            nn.Tanh(),
            nn.Linear(self.L, self.ATTENTION_BRANCHES)
        )

        # bolsa
        self.classifier = nn.Sequential(
            nn.Linear(self.M*self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

        # --- NUEVO: cabeza por instancia (parche) ---
        self.inst_classifier = nn.Sequential(
            nn.Linear(self.M, 1),
            nn.Sigmoid()
        )

        self.bce = nn.BCELoss()

    def forward(self, x):
        # x: [1,K,1,28,28] -> [K,1,28,28]
        x = x.squeeze(0)
        if x.dim() == 5 and x.shape[2] == 1:
            x = x.squeeze(2)

        H = self.feature_extractor_part1(x)
        H = H.view(-1, 50 * 4 * 4)
        H = self.feature_extractor_part2(H)          # [K,M]

        # --- NUEVO: prob por instancia ---
        inst_prob = self.inst_classifier(H).squeeze(-1)  # [K]

        A = self.attention(H)                          # [K,B]
        A = torch.transpose(A, 1, 0)                   # [B,K]
        A = F.softmax(A, dim=1)                        # sobre K

        Z = torch.mm(A, H)                             # [B,M]
        Y_prob = self.classifier(Z)                    # [B,1]
        Y_hat = torch.ge(Y_prob, 0.5).float()

        return Y_prob, Y_hat, A, inst_prob

    # --- NUEVO: objetivo para segmentación semántica (coarse por parches) ---
    def calculate_objective_seg(self, X, Y_bag, Y_inst, lambda_inst=1.0):
        """
        X: [1,K,1,28,28]
        Y_bag: [1] o [1,1]
        Y_inst: [K]
        """
        Y_bag = Y_bag.float().view(1, 1)
        Y_inst = Y_inst.float().view(-1)  # [K]

        Y_prob, _, A, inst_prob = self.forward(X)
        Y_prob = torch.clamp(Y_prob, min=1e-5, max=1.-1e-5)
        inst_prob = torch.clamp(inst_prob, min=1e-5, max=1.-1e-5)

        loss_bag = self.bce(Y_prob, Y_bag)
        loss_inst = self.bce(inst_prob, Y_inst)

        loss = loss_bag + lambda_inst * loss_inst
        return loss, A, inst_prob, Y_prob

    def calculate_classification_error(self, X, Y):
        Y = Y.float().view(1, 1)
        _, Y_hat, _, _ = self.forward(X)
        error = 1. - Y_hat.eq(Y).cpu().float().mean().item()
        return error, Y_hat

class GatedAttention(nn.Module):
    def __init__(self):
        super(GatedAttention, self).__init__()
        self.M = 500
        self.L = 128
        self.ATTENTION_BRANCHES = 1

        self.feature_extractor_part1 = nn.Sequential(
            nn.Conv2d(1, 20, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2),
            nn.Conv2d(20, 50, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2)
        )

        self.feature_extractor_part2 = nn.Sequential(
            nn.Linear(50 * 4 * 4, self.M),
            nn.ReLU(),
        )

        self.attention_V = nn.Sequential(nn.Linear(self.M, self.L), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(self.M, self.L), nn.Sigmoid())
        self.attention_w = nn.Linear(self.L, self.ATTENTION_BRANCHES)

        self.classifier = nn.Sequential(
            nn.Linear(self.M*self.ATTENTION_BRANCHES, 1),
            nn.Sigmoid()
        )

        # NUEVO
        self.inst_classifier = nn.Sequential(nn.Linear(self.M, 1), nn.Sigmoid())
        self.bce = nn.BCELoss()

    def forward(self, x):
        x = x.squeeze(0)
        if x.dim() == 5 and x.shape[2] == 1:
            x = x.squeeze(2)

        H = self.feature_extractor_part1(x)
        H = H.view(-1, 50 * 4 * 4)
        H = self.feature_extractor_part2(H)          # [K,M]

        inst_prob = self.inst_classifier(H).squeeze(-1)  # [K]

        A_V = self.attention_V(H)                    # [K,L]
        A_U = self.attention_U(H)                    # [K,L]
        A = self.attention_w(A_V * A_U)              # [K,B]
        A = torch.transpose(A, 1, 0)                 # [B,K]
        A = F.softmax(A, dim=1)

        Z = torch.mm(A, H)                           # [B,M]
        Y_prob = self.classifier(Z)                  # [B,1]
        Y_hat = torch.ge(Y_prob, 0.5).float()
        return Y_prob, Y_hat, A, inst_prob

    def calculate_objective_seg(self, X, Y_bag, Y_inst, lambda_inst=1.0):
        Y_bag = Y_bag.float().view(1, 1)
        Y_inst = Y_inst.float().view(-1)

        Y_prob, _, A, inst_prob = self.forward(X)
        Y_prob = torch.clamp(Y_prob, min=1e-5, max=1.-1e-5)
        inst_prob = torch.clamp(inst_prob, min=1e-5, max=1.-1e-5)

        loss_bag = self.bce(Y_prob, Y_bag)
        loss_inst = self.bce(inst_prob, Y_inst)

        loss = loss_bag + lambda_inst * loss_inst
        return loss, A, inst_prob, Y_prob

    def calculate_classification_error(self, X, Y):
        Y = Y.float().view(1, 1)
        _, Y_hat, _, _ = self.forward(X)
        error = 1. - Y_hat.eq(Y).cpu().float().mean().item()
        return error, Y_hat

from torch.autograd import Variable

def train(epoch, lambda_inst=1.0):
    model.train()
    train_loss = 0.0
    total, correct_bag = 0, 0

    # acumuladores segmentación
    TP, FP, FN, TN = 0, 0, 0, 0
    correct_inst, total_inst = 0, 0

    for batch_idx, (data, label) in enumerate(train_loader):
        bag_label = label[0]          # [1]
        inst_labels = label[1]        # [K]
        if args.cuda:
            data = data.cuda()
            bag_label = bag_label.cuda()
            inst_labels = inst_labels.cuda()

        data = Variable(data)
        bag_label = Variable(bag_label)
        inst_labels = Variable(inst_labels)

        optimizer.zero_grad()

        loss, A, inst_prob, Y_prob = model.calculate_objective_seg(
            data, bag_label, inst_labels, lambda_inst=lambda_inst
        )

        train_loss += loss.item()
        loss.backward()
        optimizer.step()

        # ---- métricas de bolsa ----
        pred_bag = (Y_prob >= 0.5).float()
        correct_bag += (pred_bag.view_as(bag_label.float()) == bag_label.float()).sum().item()
        total += 1

        # ---- métricas de instancia (segmentación) ----
        pred_inst = (inst_prob >= 0.5).float()          # [K]
        true_inst = inst_labels.view(-1).float()        # [K]

        correct_inst += (pred_inst == true_inst).sum().item()
        total_inst += true_inst.numel()

        # confusion
        y_true = true_inst.detach().cpu().numpy().astype(int)
        y_pred = pred_inst.detach().cpu().numpy().astype(int)
        tp, fp, fn, tn = confusion_from_binary(y_true, y_pred)
        TP += tp; FP += fp; FN += fn; TN += tn

    train_loss /= len(train_loader)
    bag_acc = correct_bag / max(1, total)
    inst_acc = correct_inst / max(1, total_inst)

    iou = iou_score(TP, FP, FN)
    dice = dice_score(TP, FP, FN)
    prec = precision_score(TP, FP, FN, TN)
    rec = recall_score(TP, FP, FN, TN)
    spec = specificity_score(TP, FP, FN, TN)
    bacc = balanced_accuracy(TP, FP, FN, TN)

    print(
        f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
        f"bag_acc={bag_acc:.3f} | inst_acc={inst_acc:.3f} | "
        f"IoU={iou:.3f} | Dice={dice:.3f} | Prec={prec:.3f} | Rec={rec:.3f} | "
        f"Spec={spec:.3f} | BAcc={bacc:.3f}"
    )


def test(lambda_inst=1.0):
    model.eval()
    test_loss = 0.0
    total, correct_bag = 0, 0

    TP, FP, FN, TN = 0, 0, 0, 0
    correct_inst, total_inst = 0, 0

    with torch.no_grad():
        for batch_idx, (data, label) in enumerate(test_loader):
            bag_label = label[0]
            inst_labels = label[1]
            if args.cuda:
                data = data.cuda()
                bag_label = bag_label.cuda()
                inst_labels = inst_labels.cuda()

            loss, A, inst_prob, Y_prob = model.calculate_objective_seg(
                data, bag_label, inst_labels, lambda_inst=lambda_inst
            )
            test_loss += loss.item()

            pred_bag = (Y_prob >= 0.5).float()
            correct_bag += (pred_bag.view_as(bag_label.float()) == bag_label.float()).sum().item()
            total += 1

            pred_inst = (inst_prob >= 0.5).float()
            true_inst = inst_labels.view(-1).float()

            correct_inst += (pred_inst == true_inst).sum().item()
            total_inst += true_inst.numel()

            y_true = true_inst.detach().cpu().numpy().astype(int)
            y_pred = pred_inst.detach().cpu().numpy().astype(int)
            tp, fp, fn, tn = confusion_from_binary(y_true, y_pred)
            TP += tp; FP += fp; FN += fn; TN += tn

    test_loss /= len(test_loader)
    bag_acc = correct_bag / max(1, total)
    inst_acc = correct_inst / max(1, total_inst)

    iou = iou_score(TP, FP, FN)
    dice = dice_score(TP, FP, FN)
    prec = precision_score(TP, FP, FN, TN)
    rec = recall_score(TP, FP, FN, TN)
    spec = specificity_score(TP, FP, FN, TN)
    bacc = balanced_accuracy(TP, FP, FN, TN)

    print(
        f"Test | loss={test_loss:.4f} | "
        f"bag_acc={bag_acc:.3f} | inst_acc={inst_acc:.3f} | "
        f"IoU={iou:.3f} | Dice={dice:.3f} | Prec={prec:.3f} | Rec={rec:.3f} | "
        f"Spec={spec:.3f} | BAcc={bacc:.3f}"
    )

# Initialize argparse and args BEFORE defining loader_kwargs
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--model', type=str, default='gated_attention', help='Choose b/w attention and gated_attention')
parser.add_argument('--no-cuda', action='store_true', default=False, help='disables CUDA training')
parser.add_argument('--seed', type=int, default=1, metavar='S', help='random seed (default: 1)')
parser.add_argument('--lr', type=float, default=0.0005, metavar='LR', help='learning rate (default: 0.0005)')
parser.add_argument('--reg', type=float, default=10e-5, metavar='R', help='weight decay')
parser.add_argument('--target_number', type=int, default=9, metavar='T', help='bags have a positive labels if they contain at least one 9')
parser.add_argument('--mean_bag_length', type=int, default=10, metavar='ML', help='average bag length')
parser.add_argument('--var_bag_length', type=int, default=2, metavar='VL', help='variance of bag length')
parser.add_argument('--num_bags_train', type=int, default=200, metavar='NTrain', help='number of bags in training set')
parser.add_argument('--num_bags_test', type=int, default=50, metavar='NTest', help='number of bags in test set')
parser.add_argument('--epochs', type=int, default=20, metavar='N', help='number of epochs to train (default: 20)')
args = parser.parse_args(args=[])
args.cuda = not args.no_cuda and torch.cuda.is_available()

torch.manual_seed(args.seed)
if args.cuda:
    torch.cuda.manual_seed(args.seed)
    print('\nGPU is ON!')

print('Load Train and Test Set (SEGMENTATION)')
loader_kwargs = {'num_workers': 1, 'pin_memory': True} if args.cuda else {}

train_loader = data_utils.DataLoader(
    MnistSegBags(grid_size=4, num_bag=args.num_bags_train, seed=args.seed, train=True, thr=0.10),
    batch_size=1, shuffle=True, **loader_kwargs
)

test_loader = data_utils.DataLoader(
    MnistSegBags(grid_size=4, num_bag=args.num_bags_test, seed=args.seed, train=False, thr=0.10),
    batch_size=1, shuffle=False, **loader_kwargs
)

print('Init Model (SEGMENTATION)')
if args.model=='attention':
    model = Attention()
elif args.model=='gated_attention':
    model = GatedAttention()

if args.cuda:
    model.cuda()

import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.999), weight_decay=args.reg)

print('Start Training (SEGMENTATION)')
lambda_inst = 1.0  # peso del término de segmentación por instancia

for epoch in range(1, args.epochs + 1):
    train(epoch, lambda_inst=lambda_inst)

print('Start Testing (SEGMENTATION)')
test(lambda_inst=lambda_inst)